Jigsaw Speed Run: 10-Min Triplet and Faiss https://www.kaggle.com/code/nghiahoangtrung/jigsaw-speed-run-10-min-triplet-and-faiss/notebook

In [1]:
%%writefile faiss_script.py
#!/usr/bin/env python3
import torch
import gc  # for garbage collection
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from sklearn.metrics.pairwise import cosine_similarity
import re
from urllib.parse import urlparse
import faiss
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import for advanced clustering
from sklearn.cluster import AgglomerativeClustering
from umap import UMAP


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))


def load_test_data():
    """Load test data."""
    print("Loading test data...")
    test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
    print(f"Loaded {len(test_df)} test examples")
    print(f"Unique rules: {test_df['rule'].nunique()}")
    return test_df


def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    
    all_texts = set()
    
    # Add all bodies
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    
    # Add all positive and negative examples
    example_cols = ['positive_example_1', 'positive_example_2', 
                   'negative_example_1', 'negative_example_2']
    
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts


def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    
    return embeddings


def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data: anchor=rule, positive=positive_example, negative=negative_example."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    anchors = []
    positives = []
    negatives = []
    
    print("Creating rule-aligned triplets from test data...")
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        
        pos_examples = []  # Will contain compliant comments (rule-aligned)
        neg_examples = []  # Will contain violating comments (rule-misaligned)

        for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))

        for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)
    
    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        
        rule_positives = {}
        rule_negatives = {}
        
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            
            pos_pool = []
            neg_pool = []
            
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))
        
        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            
            n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))
            
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))
    
    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    
    # Apply subsampling if requested
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    
    print(f"Created {len(anchors)} triplets from test data")
    
    dataset = Dataset.from_dict({
        'anchor': list(anchors),
        'positive': list(positives),
        'negative': list(negatives)
    })
    
    return dataset


def fine_tune_model(model, train_dataset, epochs=3, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss on test data."""
    
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    
    loss = TripletLoss(model=model, triplet_margin=margin)
    
    # Calculate max_steps for small datasets
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=True,
        gradient_accumulation_steps = 1,
        max_steps=max_steps,
        report_to="none"
    )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=loss,
    )
    
    trainer.train()
    
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    
    return model, final_model_path


def load_or_create_finetuned_model(test_df):
    """Load fine-tuned model if exists, otherwise create and fine-tune it."""
    
    fine_tuned_path = "./models/test-finetuned-bge/final"
    
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()
        return model
    
    print("Fine-tuned model not found. Creating new one...")
    
    print("Loading base BGE embedding model...")
    # Try Kaggle path first, fallback to HuggingFace
    try:
        #model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        #model_path = "/kaggle/input/baai/transformers/bge-large-en-v1.5/1"
        #model_path = "/kaggle/input/qwen-3-embedding/transformers/4b/1"
        model_path = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
        #model_path = "/kaggle/input/embeddinggemma/transformers/embeddinggemma-300m/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path with explicit pooling")
    except:
        model_path = ""  # BAAI/bge-small-en-v1.5
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from local path with explicit pooling")
    
    
    #triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=2, subsample_fraction=1.)
    #triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=1, subsample_fraction=1.)
    triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=1, subsample_fraction=0.35)
    
    fine_tuned_model, model_path = fine_tune_model(
        model=base_model,
        train_dataset=triplet_dataset,
        epochs=1,
        batch_size=16,
        learning_rate=2e-5,
        margin=0.25
    )
    
    print(f"Fine-tuning completed. Model saved to: {model_path}")
    fine_tuned_model.half()
    return fine_tuned_model


def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule."""
    print("Generating rule embeddings...")
    
    unique_rules = test_df['rule'].unique()
    rule_embeddings = {}
    
    for rule in unique_rules:
        clean_rule = cleaner(str(rule))
        rule_emb = model.encode(
            clean_rule,
            convert_to_tensor=False,
            normalize_embeddings=True
        )
        rule_embeddings[rule] = rule_emb
        
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings


def create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings):
    """Create centroids using Hierarchical Clustering + UMAP for better cluster representation."""
    print(f"\nCreating rule centroids with Hierarchical Clustering + UMAP...")
    
    # Initialize UMAP reducer
    umap_reducer = UMAP(n_components=32, random_state=42)  # Reduce to 32 dimensions
    
    rule_centroids = {}

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        # Collect positive examples
        pos_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])

        # Collect negative examples
        neg_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if pos_embeddings and neg_embeddings:
            pos_embeddings = np.array(pos_embeddings)
            neg_embeddings = np.array(neg_embeddings)

            # Apply UMAP to reduce dimensions before clustering
            if pos_embeddings.shape[0] > 10 and pos_embeddings.shape[0] > umap_reducer.n_components:
                pos_reduced = umap_reducer.fit_transform(pos_embeddings)
            else:
                pos_reduced = pos_embeddings
            
            if neg_embeddings.shape[0] > 10 and neg_embeddings.shape[0] > umap_reducer.n_components:
                neg_reduced = umap_reducer.fit_transform(neg_embeddings)
            else:
                neg_reduced = neg_embeddings

            # Apply Hierarchical Clustering
            n_pos_clusters = min(3, len(pos_embeddings))  # Max 3 clusters for positive
            n_neg_clusters = min(3, len(neg_embeddings))  # Max 3 clusters for negative
            
            pos_centroids = []
            neg_centroids = []

            if n_pos_clusters > 1:
                pos_clusterer = AgglomerativeClustering(n_clusters=n_pos_clusters)
                pos_labels = pos_clusterer.fit_predict(pos_reduced)
                
                # Calculate centroid for each cluster
                for cluster_id in np.unique(pos_labels):
                    cluster_mask = pos_labels == cluster_id
                    cluster_embeddings = pos_embeddings[cluster_mask]
                    cluster_centroid = cluster_embeddings.mean(axis=0)
                    cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)  # Normalize
                    pos_centroids.append(cluster_centroid)
            else:
                # If not enough data for clustering, use single centroid
                pos_centroid = pos_embeddings.mean(axis=0)
                pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
                pos_centroids.append(pos_centroid)

            if n_neg_clusters > 1:
                neg_clusterer = AgglomerativeClustering(n_clusters=n_neg_clusters)
                neg_labels = neg_clusterer.fit_predict(neg_reduced)
                
                # Calculate centroid for each cluster
                for cluster_id in np.unique(neg_labels):
                    cluster_mask = neg_labels == cluster_id
                    cluster_embeddings = neg_embeddings[cluster_mask]
                    cluster_centroid = cluster_embeddings.mean(axis=0)
                    cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)  # Normalize
                    neg_centroids.append(cluster_centroid)
            else:
                # If not enough data for clustering, use single centroid
                neg_centroid = neg_embeddings.mean(axis=0)
                neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)
                neg_centroids.append(neg_centroid)

            rule_centroids[rule] = {
                'positive_centroids': pos_centroids,
                'negative_centroids': neg_centroids,
                'pos_count': len(pos_embeddings),
                'neg_count': len(neg_embeddings),
                'rule_embedding': rule_embeddings[rule]
            }

            print(f"  Rule: {rule[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)} - Clusters: Pos={len(pos_centroids)}, Neg={len(neg_centroids)}")

    print(f"Created hierarchical centroids for {len(rule_centroids)} rules")
    return rule_centroids


def predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids):
    """Predict test set using hierarchical clustering centroids and distance metrics."""
    print("\nMaking predictions on test set with Hierarchical Clustering centroids...")

    row_ids = []
    predictions = []

    for rule in test_df['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        rule_data = test_df[test_df['rule'] == rule]

        if rule not in rule_centroids:
            continue

        pos_centroids = rule_centroids[rule]['positive_centroids']
        neg_centroids = rule_centroids[rule]['negative_centroids']

        # Process all bodies for this rule
        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            row_id = row['row_id']

            if body not in text_to_embedding:
                continue

            body_embedding = text_to_embedding[body]

            # Calculate distances to all positive centroids
            pos_distances = []
            for pos_centroid in pos_centroids:
                #distance = np.linalg.norm(body_embedding - pos_centroid)  # Euclidean distance
                distance = 1 - cosine_similarity([body_embedding], [pos_centroid])[0][0]  # Cosine distance
                pos_distances.append(distance)

            # Calculate distances to all negative centroids
            neg_distances = []
            for neg_centroid in neg_centroids:
                #distance = np.linalg.norm(body_embedding - neg_centroid)  # Euclidean distance
                distance = 1 - cosine_similarity([body_embedding], [neg_centroid])[0][0] # Cosine distance
                neg_distances.append(distance)

            # Use minimum distances (closest centroids)
            min_pos_distance = min(pos_distances) if pos_distances else 1.0
            min_neg_distance = min(neg_distances) if neg_distances else 1.0

            # Score: closer to negative (violating) = higher score
            rule_prediction = min_neg_distance - min_pos_distance

            row_ids.append(row_id)
            predictions.append(rule_prediction)

    print(f"Made predictions for {len(predictions)} test examples")
    return row_ids, np.array(predictions)


def main():
    """Main inference pipeline with Hierarchical Clustering + UMAP improvements."""
    print("="*70)
    print("IMPROVED SIMILARITY CLASSIFIER - HIERARCHICAL CLUSTERING + UMAP")
    print("="*70)
    
    # Step 1: Load test data
    test_df = load_test_data()
    
    # Step 2: Load or create fine-tuned model
    print("\n" + "="*50)
    print("MODEL PREPARATION PHASE")
    print("="*50)
    model = load_or_create_finetuned_model(test_df)
    
    # Step 3: Collect all texts
    all_texts = collect_all_texts(test_df)
    
    # Step 4: Generate embeddings with fine-tuned model
    print("\n" + "="*50)
    print("EMBEDDING GENERATION PHASE")
    print("="*50)
    all_embeddings = generate_embeddings(all_texts, model)
    
    # Step 5: Create text to embedding mapping
    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}
    
    # Step 6: Generate rule embeddings
    rule_embeddings = generate_rule_embeddings(test_df, model)
    
    # Step 7: Create rule centroids with Hierarchical Clustering + UMAP
    rule_centroids = create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings)
    
    # Step 8: Predict test set with hierarchical clustering
    print("\n" + "="*50)
    print("PREDICTION PHASE")
    print("="*50)
    row_ids, predictions = predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids)
    
    # Step 9: Create submission
    submission_df = pd.DataFrame({
        'row_id': row_ids,
        'rule_violation': predictions
    })

    # Sort by row_id
    submission_df = submission_df.sort_values(by="row_id").reset_index(drop=True)
    submission_df.to_csv('submission_faiss.csv', index=False)
    print(f"\nSaved predictions for {len(submission_df)} test examples to submission.csv")
    try:
        # Free GPU memory
        del model, all_embeddings, text_to_embedding, rule_embeddings, rule_centroids
        torch.cuda.empty_cache()
        gc.collect()
    except Exception as e:
        print('Error when free GPU memory', e)
    print(f"\n{'='*70}")
    print(f"HIERARCHICAL CLUSTERING + UMAP INFERENCE COMPLETED")
    print(f"Model: Fine-tuned BGE on test data triplets")
    print(f"Method: Hierarchical clustering + UMAP dimensionality reduction")
    print(f"Predicted on {len(test_df)} test examples")
    print(f"Prediction stats: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")
    print(f"{'='*70}")


if True:
    main()

Writing faiss_script.py


In [2]:
!python faiss_script.py

2025-10-23 01:58:05.897342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761184686.119402      46 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761184686.184103      46 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
IMPROVED SIMILARITY CLASSIFIER - HIERARCHICAL CLUSTERING + UMAP
Loading test data...
Loaded 10 test examples
Unique rules: 2

MODEL PREPARATION PHASE
Fine-tuned model not found. Creating new one...
Loading base BGE embedding model...
Loaded base model from Kaggle path with explicit pooling
Creating rule-aligned triplets from test data...
Processing test rows: 100%|███████████████████| 10/10 [00:00<00:00, 5392.52it/s]
Adding 1x augmen

End FAISS

In [3]:
# !pip install -q -U torch torchaudio torchvision transformers datasets accelerate peft trl bitsandbytes evaluate unsloth

In [4]:
import unsloth
import torch
from peft import PeftModel
from unsloth import FastLanguageModel
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import os
import random
import gc
import time
import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
from datasets import Dataset
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-23 01:59:39.226149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761184779.248177      26 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761184779.255448      26 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


# 42,43,44,45 

In [5]:
%%writefile trainer_script_42.py
# ==============================================================================
# 步骤 1: 导入所有必需的库
# ==============================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1" # 关闭联网，否则unsloth会卡住
import unsloth
from unsloth import FastLanguageModel
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from peft import PeftModel
from tqdm import tqdm
import gc
import torch.nn.functional as F
import random

SEED = 3100

# ==============================================================================
#  ⭐ 新增：设置全局随机种子以保证可复现性 ⭐
# ==============================================================================
def set_seed(seed_value=SEED):
    """设置所有相关的随机种子"""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # if you are using multi-GPU.

# 调用函数
set_seed()

# ==============================================================================
#  ⭐ 新增：设置CUDA操作为确定性模式 ⭐
# ==============================================================================
# 这会牺牲一些性能，但对于复现性至关重要
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==============================================================================
# 步骤 2: 定义常量和配置
# ==============================================================================
# 定义输入/输出路径
BASE_MODEL_PATH = "/kaggle/input/qwen3-8b-unsloth-bnb-4bit/transformers/default/1" # 7:45
BASE_MODEL_PATH = "/kaggle/input/llama-3-8b-instruct-bnb-4bit/transformers/default/1/llama-3" # 7:45
BASE_MODEL_PATH = "/kaggle/input/unslothgemma-2-9b-bnb-4bit/tensorflow2/default/1/gemma-2-9b-bnb-4bit" # 12:41
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit" # 17
# BASE_MODEL_PATH = "/kaggle/input/unslothqwen2.5-14b-bnb-4bit/tensorflow2/default/1/Qwen2.5-14B-bnb-4bit" # 暂时无法运行后续找找原因，好像是先加载到GPU:0上再搬到GPU:1上导致00M
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit" # 15
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
# DATA_PATH = "/kaggle/working/"
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.csv")
# 统一的模型输出和提交文件路径
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_42/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission.csv"
# 确保输出目录存在
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# ==============================================================================
# 3. 定义所有辅助函数
# ==============================================================================
class ClassificationModel(torch.nn.Module): # 参考AutoModelForSequenceClassification的实现，保持一致
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.base_model = base_model
        self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 步骤 1: 调用基础模型获取最后一层的隐藏状态
        # `outputs.hidden_states` 只有在 `output_hidden_states=True` 时才会返回
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        # 步骤 2: 智能定位最后一个有效 token 的索引
        # 获取基础模型（原始的、未被PEFT包装的模型）的 pad_token_id
        # PeftModel 会自动将 config 传递过来
        pad_token_id = self.base_model.config.pad_token_id
      
        # 创建一个掩码，标记出所有非填充 token
        # .ne() 是 "not equal" 的缩写
        non_pad_mask = input_ids.ne(pad_token_id)
      
        # 计算每个序列的实际长度，并找到最后一个有效 token 的索引
        # .sum(dim=1) 计算每行的 True (即 1) 的数量
        # 减 1 是因为索引是从 0 开始的
        last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
      
        # 步骤 3: 使用计算出的索引来提取每个序列对应的隐藏状态
        batch_size = input_ids.shape[0]
        # 使用 torch.gather 沿着 hidden_size 维度精确提取
        # 首先，我们需要将索引张量调整为正确的形状以进行 gather 操作
        # 形状需要是 (batch_size, 1, hidden_size) 以便从 (batch_size, seq_len, hidden_size) 中提取
        expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
      
        # 使用 gather 从 hidden_states 中提取出每个序列最后一个有效 token 的表征
        pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
        # 步骤 4: 将提取出的表征送入分类头
        logits = self.classifier(pooled_hidden_states)
      
        # 步骤 5: 计算损失（如果提供了标签）并返回标准输出
        output = {"logits": logits}
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            output["loss"] = loss
          
        return output
# ==============================================================================
# 步骤 3: 定义通用的数据处理函数
# ==============================================================================
def create_classifier_input(row):
    """为分类任务创建一个更明确、更丰富的输入文本"""
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
def get_dataframe_to_train(data_path):
    """
    加载、合并和准备训练数据，包括从测试集中提取的样本。
    对于重复的 (body, rule) 组合，通过投票选择出现次数最多的标签。
    """
    train_dataset = pd.read_csv(os.path.join(data_path, "train.csv"))
    test_dataset = pd.read_csv(os.path.join(data_path, "test.csv"))
    flatten = []
    # 保留原始训练集
    train_subset = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_subset = train_subset.rename(columns={"rule_violation": "label"})
    flatten.append(train_subset)
  
    # 从测试集中提取带标签的样本
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["label"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)
    # 丢弃任何关键字段为空的行
    dataframe = dataframe.dropna(subset=['body', 'rule', 'label'])
  
    # --- 核心修改部分 ---
    # 步骤 1: 计算每个 (body, rule, label) 组合的出现次数
    counts = dataframe.groupby(['body', 'rule', 'label']).size().reset_index(name='counts')
  
    # 步骤 2: 找出每个 (body, rule) 组合中，出现次数最多的那个标签
    # 首先按 (body, rule) 分组，然后找到每组中 'counts' 最大值的索引
    idx = counts.groupby(['body', 'rule'])['counts'].transform(max) == counts['counts']
  
    # 使用该索引筛选出最常见的组合
    # 如果有多个标签出现次数相同，.drop_duplicates会保留第一个
    most_common_df = counts[idx].drop_duplicates(subset=['body', 'rule'], keep='first')
  
    # 步骤 3: 从原始 DataFrame 中筛选出 subreddit 信息并合并
    # 我们需要保留 subreddit 列，但它不在我们的分组键中
    # 先对原始数据去重，以防一个样本有多个 subreddit
    subreddit_info = dataframe[['body', 'rule', 'subreddit']].drop_duplicates(subset=['body', 'rule'])
  
    # 将最常见的标签与 subreddit 信息合并
    final_df = pd.merge(most_common_df[['body', 'rule', 'label']], subreddit_info, on=['body', 'rule'], how='left')

    # 新增: 增加额外的补充数据集
    # 1. 根据train.csv和test.csv中的rule进行去重，获得rules列表
    rules = pd.concat([train_dataset['rule'], test_dataset['rule']]).unique()

    # 2. 获得/kaggle/input/extra-datas/extra.csv的去重后的rule，获得rules_extra列表
    extra_df = pd.read_csv("/kaggle/input/1003-extra-data03/extra_translated_output.csv", encoding='utf-8', encoding_errors='replace')

    rules_extra = extra_df['rule'].unique()

    # 3. 将rules和rules_extra转成小写后进行匹配（仅匹配前5个字符）
    rules_lower = {r.lower()[:5]: r for r in rules}

    # 4. 将extra.csv中的rule_extra通过匹配，得到真正的rule，然后替换
    rule_map = {}
    for extra_r in rules_extra:
        key = extra_r.lower()[:5]
        rule_map[extra_r] = rules_lower[key]
    extra_df['rule'] = extra_df['rule'].map(rule_map)
    extra_df = extra_df.rename(columns={'rule_violation': 'label'})

    # 5. 将extra.csv加入final_df
    final_df = pd.concat([final_df, extra_df[['body', 'rule', 'label', 'subreddit']]], ignore_index=True)

    final_df.to_csv('/kaggle/working/final_df.csv', index=False)
    
    return final_df
# ==============================================================================
# 步骤 4: 定义主训练函数
# ==============================================================================
def run_training():
    """
    执行完整的训练流程。
    """
    print("="*50)
    print(" " * 20 + "开始训练流程")
    print("="*50)
  
    print("\n--- 步骤 4.1: 加载 Tokenizer 和模型 ---")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_PATH,
        max_seq_length=256,
        load_in_4bit=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = ClassificationModel(base_model, 2)
    print("\n--- 步骤 4.2: 配置 LoRA ---")
    model.base_model = FastLanguageModel.get_peft_model(
        model.base_model,
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj" , "all-linear"],
    )
    print("LoRA 配置完成。")
    print("用哑层替换 lm_head 以节省显存并兼容 Unsloth...")
    original_model = model.base_model.get_base_model()
    hidden_size = original_model.config.hidden_size
  
    # 1. 创建一个新的、极小的线性层作为哑层
    dummy_lm_head = torch.nn.Linear(hidden_size, 1, bias=False)
  
    # 2. 明确地将这个哑层的参数设置为不可训练
    for param in dummy_lm_head.parameters():
        param.requires_grad = False
  
    # 3. 将这个不可训练的哑层赋给模型
    original_model.lm_head = dummy_lm_head
    # print(model)
    print("\n--- 步骤 4.3: 准备训练数据集 ---")
    dataframe = get_dataframe_to_train(DATA_PATH)
    dataframe["text"] = dataframe.apply(create_classifier_input, axis=1)
    dataframe['label'] = dataframe['label'].astype(int)
    dataset = Dataset.from_pandas(dataframe[["text", "label"]])
  
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
    train_dataset = dataset.map(tokenize_function, batched=True)
    train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])
    print(f"数据集准备完毕，总计 {len(train_dataset)} 条训练样本。")
    print("\n--- 步骤 4.4: 设置训练参数 ---")
    training_args = TrainingArguments(
        output_dir=MODEL_OUTPUT_PATH,
        num_train_epochs=2,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        remove_unused_columns=True,
        report_to="none",
        torch_compile=True,
        ddp_find_unused_parameters=False,
        seed=SEED,
        group_by_length=True
    )
    print("\n--- 步骤 4.5: 初始化并开始训练 ---")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()
    print("\n--- 训练完成！ ---")
    print(f"\n--- 步骤 4.6: 保存适配器权重到 {MODEL_OUTPUT_PATH} ---")
    model.base_model.save_pretrained(MODEL_OUTPUT_PATH)
    torch.save(model.classifier.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "classifier.pth"))
    tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
    print("模型已成功保存。")
# ==============================================================================
# 步骤 6: 运行完整的训练和推理流程
# ==============================================================================
if __name__ == "__main__" or "kaggle.python" in str(get_ipython()):
    # 读取测试文件以检查其行数
    test_df_check = pd.read_csv(TEST_FILE_PATH)
  
    # 检查行数是否小于100
    if len(test_df_check) < 100:
        print("测试文件行数小于100，判定为测试运行。")
        print("将直接生成一个空的 submission.csv 文件。")
      
        # 创建一个包含 'row_id' 和 'rule_violation' 列的空 DataFrame
        # 'row_id' 来自于我们读取的测试文件
        submission_df = pd.DataFrame({
            'row_id': test_df_check['row_id'],
            'rule_violation': 0.5 # 通常用0.5作为默认值
        })
      
        # 保存为空的提交文件
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
      
        print(f"空的提交文件 '{SUBMISSION_FILE_PATH}' 已生成。")
        print("文件预览:")
        print(submission_df.head())
      
    else:
        print("测试文件行数大于等于100，开始完整的训练和推理流程。")
      
        # 第一步：训练模型
        run_training()

Writing trainer_script_42.py


In [6]:
%%writefile trainer_script_43.py
# ==============================================================================
# 步骤 1: 导入所有必需的库
# ==============================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1" # 关闭联网，否则unsloth会卡住
import unsloth
from unsloth import FastLanguageModel
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from peft import PeftModel
from tqdm import tqdm
import gc
import torch.nn.functional as F
import random

SEED = 3043

# ==============================================================================
#  ⭐ 新增：设置全局随机种子以保证可复现性 ⭐
# ==============================================================================
def set_seed(seed_value=SEED):
    """设置所有相关的随机种子"""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # if you are using multi-GPU.

# 调用函数
set_seed()

# ==============================================================================
#  ⭐ 新增：设置CUDA操作为确定性模式 ⭐
# ==============================================================================
# 这会牺牲一些性能，但对于复现性至关重要
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==============================================================================
# 步骤 2: 定义常量和配置
# ==============================================================================
# 定义输入/输出路径
BASE_MODEL_PATH = "/kaggle/input/qwen3-8b-unsloth-bnb-4bit/transformers/default/1" # 7:45
BASE_MODEL_PATH = "/kaggle/input/llama-3-8b-instruct-bnb-4bit/transformers/default/1/llama-3" # 7:45
BASE_MODEL_PATH = "/kaggle/input/unslothgemma-2-9b-bnb-4bit/tensorflow2/default/1/gemma-2-9b-bnb-4bit" # 12:41
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit" # 17
# BASE_MODEL_PATH = "/kaggle/input/unslothqwen2.5-14b-bnb-4bit/tensorflow2/default/1/Qwen2.5-14B-bnb-4bit" # 暂时无法运行后续找找原因，好像是先加载到GPU:0上再搬到GPU:1上导致00M
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit" # 15
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
# DATA_PATH = "/kaggle/working/"
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.csv")
# 统一的模型输出和提交文件路径
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_43/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission.csv"
# 确保输出目录存在
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# ==============================================================================
# 3. 定义所有辅助函数
# ==============================================================================
class ClassificationModel(torch.nn.Module): # 参考AutoModelForSequenceClassification的实现，保持一致
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.base_model = base_model
        self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 步骤 1: 调用基础模型获取最后一层的隐藏状态
        # `outputs.hidden_states` 只有在 `output_hidden_states=True` 时才会返回
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        # 步骤 2: 智能定位最后一个有效 token 的索引
        # 获取基础模型（原始的、未被PEFT包装的模型）的 pad_token_id
        # PeftModel 会自动将 config 传递过来
        pad_token_id = self.base_model.config.pad_token_id
      
        # 创建一个掩码，标记出所有非填充 token
        # .ne() 是 "not equal" 的缩写
        non_pad_mask = input_ids.ne(pad_token_id)
      
        # 计算每个序列的实际长度，并找到最后一个有效 token 的索引
        # .sum(dim=1) 计算每行的 True (即 1) 的数量
        # 减 1 是因为索引是从 0 开始的
        last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
      
        # 步骤 3: 使用计算出的索引来提取每个序列对应的隐藏状态
        batch_size = input_ids.shape[0]
        # 使用 torch.gather 沿着 hidden_size 维度精确提取
        # 首先，我们需要将索引张量调整为正确的形状以进行 gather 操作
        # 形状需要是 (batch_size, 1, hidden_size) 以便从 (batch_size, seq_len, hidden_size) 中提取
        expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
      
        # 使用 gather 从 hidden_states 中提取出每个序列最后一个有效 token 的表征
        pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
        # 步骤 4: 将提取出的表征送入分类头
        logits = self.classifier(pooled_hidden_states)
      
        # 步骤 5: 计算损失（如果提供了标签）并返回标准输出
        output = {"logits": logits}
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            output["loss"] = loss
          
        return output
# ==============================================================================
# 步骤 3: 定义通用的数据处理函数
# ==============================================================================
def create_classifier_input(row):
    """为分类任务创建一个更明确、更丰富的输入文本"""
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
def get_dataframe_to_train(data_path):
    """
    加载、合并和准备训练数据，包括从测试集中提取的样本。
    对于重复的 (body, rule) 组合，通过投票选择出现次数最多的标签。
    """
    train_dataset = pd.read_csv(os.path.join(data_path, "train.csv"))
    test_dataset = pd.read_csv(os.path.join(data_path, "test.csv"))
    flatten = []
    # 保留原始训练集
    train_subset = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_subset = train_subset.rename(columns={"rule_violation": "label"})
    flatten.append(train_subset)
  
    # 从测试集中提取带标签的样本
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["label"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)
    # 丢弃任何关键字段为空的行
    dataframe = dataframe.dropna(subset=['body', 'rule', 'label'])
  
    # --- 核心修改部分 ---
    # 步骤 1: 计算每个 (body, rule, label) 组合的出现次数
    counts = dataframe.groupby(['body', 'rule', 'label']).size().reset_index(name='counts')
  
    # 步骤 2: 找出每个 (body, rule) 组合中，出现次数最多的那个标签
    # 首先按 (body, rule) 分组，然后找到每组中 'counts' 最大值的索引
    idx = counts.groupby(['body', 'rule'])['counts'].transform(max) == counts['counts']
  
    # 使用该索引筛选出最常见的组合
    # 如果有多个标签出现次数相同，.drop_duplicates会保留第一个
    most_common_df = counts[idx].drop_duplicates(subset=['body', 'rule'], keep='first')
  
    # 步骤 3: 从原始 DataFrame 中筛选出 subreddit 信息并合并
    # 我们需要保留 subreddit 列，但它不在我们的分组键中
    # 先对原始数据去重，以防一个样本有多个 subreddit
    subreddit_info = dataframe[['body', 'rule', 'subreddit']].drop_duplicates(subset=['body', 'rule'])
  
    # 将最常见的标签与 subreddit 信息合并
    final_df = pd.merge(most_common_df[['body', 'rule', 'label']], subreddit_info, on=['body', 'rule'], how='left')

    # 新增: 增加额外的补充数据集
    # 1. 根据train.csv和test.csv中的rule进行去重，获得rules列表
    rules = pd.concat([train_dataset['rule'], test_dataset['rule']]).unique()

    # 2. 获得/kaggle/input/extra-datas/extra.csv的去重后的rule，获得rules_extra列表
    extra_df = pd.read_csv("/kaggle/input/1003-extra-data03/extra_translated_output.csv", encoding='utf-8', encoding_errors='replace')

    rules_extra = extra_df['rule'].unique()

    # 3. 将rules和rules_extra转成小写后进行匹配（仅匹配前5个字符）
    rules_lower = {r.lower()[:5]: r for r in rules}

    # 4. 将extra.csv中的rule_extra通过匹配，得到真正的rule，然后替换
    rule_map = {}
    for extra_r in rules_extra:
        key = extra_r.lower()[:5]
        rule_map[extra_r] = rules_lower[key]
    extra_df['rule'] = extra_df['rule'].map(rule_map)
    extra_df = extra_df.rename(columns={'rule_violation': 'label'})

    # 5. 将extra.csv加入final_df
    final_df = pd.concat([final_df, extra_df[['body', 'rule', 'label', 'subreddit']]], ignore_index=True)

    final_df.to_csv('/kaggle/working/final_df.csv', index=False)
    
    return final_df
# ==============================================================================
# 步骤 4: 定义主训练函数
# ==============================================================================
def run_training():
    """
    执行完整的训练流程。
    """
    print("="*50)
    print(" " * 20 + "开始训练流程")
    print("="*50)
  
    print("\n--- 步骤 4.1: 加载 Tokenizer 和模型 ---")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_PATH,
        max_seq_length=256,
        load_in_4bit=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = ClassificationModel(base_model, 2)
    print("\n--- 步骤 4.2: 配置 LoRA ---")
    model.base_model = FastLanguageModel.get_peft_model(
        model.base_model,
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj" , "all-linear"],
    )
    print("LoRA 配置完成。")
    print("用哑层替换 lm_head 以节省显存并兼容 Unsloth...")
    original_model = model.base_model.get_base_model()
    hidden_size = original_model.config.hidden_size
  
    # 1. 创建一个新的、极小的线性层作为哑层
    dummy_lm_head = torch.nn.Linear(hidden_size, 1, bias=False)
  
    # 2. 明确地将这个哑层的参数设置为不可训练
    for param in dummy_lm_head.parameters():
        param.requires_grad = False
  
    # 3. 将这个不可训练的哑层赋给模型
    original_model.lm_head = dummy_lm_head
    # print(model)
    print("\n--- 步骤 4.3: 准备训练数据集 ---")
    dataframe = get_dataframe_to_train(DATA_PATH)
    dataframe["text"] = dataframe.apply(create_classifier_input, axis=1)
    dataframe['label'] = dataframe['label'].astype(int)
    dataset = Dataset.from_pandas(dataframe[["text", "label"]])
  
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
    train_dataset = dataset.map(tokenize_function, batched=True)
    train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])
    print(f"数据集准备完毕，总计 {len(train_dataset)} 条训练样本。")
    print("\n--- 步骤 4.4: 设置训练参数 ---")
    training_args = TrainingArguments(
        output_dir=MODEL_OUTPUT_PATH,
        num_train_epochs=2,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        remove_unused_columns=True,
        report_to="none",
        torch_compile=True,
        ddp_find_unused_parameters=False,
        seed=SEED,
        group_by_length=True
    )
    print("\n--- 步骤 4.5: 初始化并开始训练 ---")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()
    print("\n--- 训练完成！ ---")
    print(f"\n--- 步骤 4.6: 保存适配器权重到 {MODEL_OUTPUT_PATH} ---")
    model.base_model.save_pretrained(MODEL_OUTPUT_PATH)
    torch.save(model.classifier.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "classifier.pth"))
    tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
    print("模型已成功保存。")
# ==============================================================================
# 步骤 6: 运行完整的训练和推理流程
# ==============================================================================
if __name__ == "__main__" or "kaggle.python" in str(get_ipython()):
    # 读取测试文件以检查其行数
    test_df_check = pd.read_csv(TEST_FILE_PATH)
  
    # 检查行数是否小于100
    if len(test_df_check) < 100:
        print("测试文件行数小于100，判定为测试运行。")
        print("将直接生成一个空的 submission.csv 文件。")
      
        # 创建一个包含 'row_id' 和 'rule_violation' 列的空 DataFrame
        # 'row_id' 来自于我们读取的测试文件
        submission_df = pd.DataFrame({
            'row_id': test_df_check['row_id'],
            'rule_violation': 0.5 # 通常用0.5作为默认值
        })
      
        # 保存为空的提交文件
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
      
        print(f"空的提交文件 '{SUBMISSION_FILE_PATH}' 已生成。")
        print("文件预览:")
        print(submission_df.head())
      
    else:
        print("测试文件行数大于等于100，开始完整的训练和推理流程。")
      
        # 第一步：训练模型
        run_training()

Writing trainer_script_43.py


In [7]:
import subprocess
import os
import sys

# 定义两个训练任务的脚本和日志文件
tasks = {
    'GPU 0': {
        'script': 'trainer_script_42.py',
        'log': '/kaggle/working/trainer_script_42.log',
        'env': {'CUDA_VISIBLE_DEVICES': '0'}
    },
    'GPU 1': {
        'script': 'trainer_script_43.py',
        'log': '/kaggle/working/trainer_script_43.log',
        'env': {'CUDA_VISIBLE_DEVICES': '1'}
    }
}

processes = []
log_files = {}

# 循环启动所有训练任务
for device, config in tasks.items():
    # 准备环境，继承当前环境并更新GPU设置
    env = os.environ.copy()
    env.update(config['env'])
    
    # 构建命令
    command = [sys.executable, config['script']]
    
    # 打开日志文件
    log_file = open(config['log'], 'w')
    log_files[device] = log_file
    
    print(f"Starting training for {device} on script {config['script']}...")
    # 使用 Popen 启动非阻塞的后台进程
    p = subprocess.Popen(command, stdout=log_file, stderr=log_file, env=env)
    processes.append(p)

# 等待所有进程完成
print("\nBoth training processes are running in the background. Waiting for them to complete...")
for p in processes:
    p.wait()

# 关闭所有日志文件
for f in log_files.values():
    f.close()

print("\nAll training processes have finished. Check the log files for details:")
for device, config in tasks.items():
    print(f"- {device}: {config['log']}")

Starting training for GPU 0 on script trainer_script_42.py...
Starting training for GPU 1 on script trainer_script_43.py...

Both training processes are running in the background. Waiting for them to complete...

All training processes have finished. Check the log files for details:
- GPU 0: /kaggle/working/trainer_script_42.log
- GPU 1: /kaggle/working/trainer_script_43.log


In [8]:
%%writefile trainer_script_44.py
# ==============================================================================
# 步骤 1: 导入所有必需的库
# ==============================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1" # 关闭联网，否则unsloth会卡住
import unsloth
from unsloth import FastLanguageModel
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from peft import PeftModel
from tqdm import tqdm
import gc
import torch.nn.functional as F
import random

SEED = 3044

# ==============================================================================
#  ⭐ 新增：设置全局随机种子以保证可复现性 ⭐
# ==============================================================================
def set_seed(seed_value=SEED):
    """设置所有相关的随机种子"""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # if you are using multi-GPU.

# 调用函数
set_seed()

# ==============================================================================
#  ⭐ 新增：设置CUDA操作为确定性模式 ⭐
# ==============================================================================
# 这会牺牲一些性能，但对于复现性至关重要
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==============================================================================
# 步骤 2: 定义常量和配置
# ==============================================================================
# 定义输入/输出路径
BASE_MODEL_PATH = "/kaggle/input/qwen3-8b-unsloth-bnb-4bit/transformers/default/1" # 7:45
BASE_MODEL_PATH = "/kaggle/input/llama-3-8b-instruct-bnb-4bit/transformers/default/1/llama-3" # 7:45
BASE_MODEL_PATH = "/kaggle/input/unslothgemma-2-9b-bnb-4bit/tensorflow2/default/1/gemma-2-9b-bnb-4bit" # 12:41
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit" # 17
# BASE_MODEL_PATH = "/kaggle/input/unslothqwen2.5-14b-bnb-4bit/tensorflow2/default/1/Qwen2.5-14B-bnb-4bit" # 暂时无法运行后续找找原因，好像是先加载到GPU:0上再搬到GPU:1上导致00M
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit" # 15
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
# DATA_PATH = "/kaggle/working/"
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.csv")
# 统一的模型输出和提交文件路径
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_44/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission.csv"
# 确保输出目录存在
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# ==============================================================================
# 3. 定义所有辅助函数
# ==============================================================================
class ClassificationModel(torch.nn.Module): # 参考AutoModelForSequenceClassification的实现，保持一致
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.base_model = base_model
        self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 步骤 1: 调用基础模型获取最后一层的隐藏状态
        # `outputs.hidden_states` 只有在 `output_hidden_states=True` 时才会返回
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        # 步骤 2: 智能定位最后一个有效 token 的索引
        # 获取基础模型（原始的、未被PEFT包装的模型）的 pad_token_id
        # PeftModel 会自动将 config 传递过来
        pad_token_id = self.base_model.config.pad_token_id
      
        # 创建一个掩码，标记出所有非填充 token
        # .ne() 是 "not equal" 的缩写
        non_pad_mask = input_ids.ne(pad_token_id)
      
        # 计算每个序列的实际长度，并找到最后一个有效 token 的索引
        # .sum(dim=1) 计算每行的 True (即 1) 的数量
        # 减 1 是因为索引是从 0 开始的
        last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
      
        # 步骤 3: 使用计算出的索引来提取每个序列对应的隐藏状态
        batch_size = input_ids.shape[0]
        # 使用 torch.gather 沿着 hidden_size 维度精确提取
        # 首先，我们需要将索引张量调整为正确的形状以进行 gather 操作
        # 形状需要是 (batch_size, 1, hidden_size) 以便从 (batch_size, seq_len, hidden_size) 中提取
        expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
      
        # 使用 gather 从 hidden_states 中提取出每个序列最后一个有效 token 的表征
        pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
        # 步骤 4: 将提取出的表征送入分类头
        logits = self.classifier(pooled_hidden_states)
      
        # 步骤 5: 计算损失（如果提供了标签）并返回标准输出
        output = {"logits": logits}
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            output["loss"] = loss
          
        return output
# ==============================================================================
# 步骤 3: 定义通用的数据处理函数
# ==============================================================================
def create_classifier_input(row):
    """为分类任务创建一个更明确、更丰富的输入文本"""
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
def get_dataframe_to_train(data_path):
    """
    加载、合并和准备训练数据，包括从测试集中提取的样本。
    对于重复的 (body, rule) 组合，通过投票选择出现次数最多的标签。
    """
    train_dataset = pd.read_csv(os.path.join(data_path, "train.csv"))
    test_dataset = pd.read_csv(os.path.join(data_path, "test.csv"))
    flatten = []
    # 保留原始训练集
    train_subset = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_subset = train_subset.rename(columns={"rule_violation": "label"})
    flatten.append(train_subset)
  
    # 从测试集中提取带标签的样本
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["label"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)
    # 丢弃任何关键字段为空的行
    dataframe = dataframe.dropna(subset=['body', 'rule', 'label'])
  
    # --- 核心修改部分 ---
    # 步骤 1: 计算每个 (body, rule, label) 组合的出现次数
    counts = dataframe.groupby(['body', 'rule', 'label']).size().reset_index(name='counts')
  
    # 步骤 2: 找出每个 (body, rule) 组合中，出现次数最多的那个标签
    # 首先按 (body, rule) 分组，然后找到每组中 'counts' 最大值的索引
    idx = counts.groupby(['body', 'rule'])['counts'].transform(max) == counts['counts']
  
    # 使用该索引筛选出最常见的组合
    # 如果有多个标签出现次数相同，.drop_duplicates会保留第一个
    most_common_df = counts[idx].drop_duplicates(subset=['body', 'rule'], keep='first')
  
    # 步骤 3: 从原始 DataFrame 中筛选出 subreddit 信息并合并
    # 我们需要保留 subreddit 列，但它不在我们的分组键中
    # 先对原始数据去重，以防一个样本有多个 subreddit
    subreddit_info = dataframe[['body', 'rule', 'subreddit']].drop_duplicates(subset=['body', 'rule'])
  
    # 将最常见的标签与 subreddit 信息合并
    final_df = pd.merge(most_common_df[['body', 'rule', 'label']], subreddit_info, on=['body', 'rule'], how='left')

    # 新增: 增加额外的补充数据集
    # 1. 根据train.csv和test.csv中的rule进行去重，获得rules列表
    rules = pd.concat([train_dataset['rule'], test_dataset['rule']]).unique()

    # 2. 获得/kaggle/input/extra-datas/extra.csv的去重后的rule，获得rules_extra列表
    extra_df = pd.read_csv("/kaggle/input/1003-extra-data03/extra_translated_output.csv", encoding='utf-8', encoding_errors='replace')

    rules_extra = extra_df['rule'].unique()

    # 3. 将rules和rules_extra转成小写后进行匹配（仅匹配前5个字符）
    rules_lower = {r.lower()[:5]: r for r in rules}

    # 4. 将extra.csv中的rule_extra通过匹配，得到真正的rule，然后替换
    rule_map = {}
    for extra_r in rules_extra:
        key = extra_r.lower()[:5]
        rule_map[extra_r] = rules_lower[key]
    extra_df['rule'] = extra_df['rule'].map(rule_map)
    extra_df = extra_df.rename(columns={'rule_violation': 'label'})

    # 5. 将extra.csv加入final_df
    # final_df = pd.concat([final_df, extra_df[['body', 'rule', 'label', 'subreddit']]], ignore_index=True)

    final_df.to_csv('/kaggle/working/final_df.csv', index=False)
    
    return final_df
# ==============================================================================
# 步骤 4: 定义主训练函数
# ==============================================================================
def run_training():
    """
    执行完整的训练流程。
    """
    print("="*50)
    print(" " * 20 + "开始训练流程")
    print("="*50)
  
    print("\n--- 步骤 4.1: 加载 Tokenizer 和模型 ---")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_PATH,
        max_seq_length=256,
        load_in_4bit=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = ClassificationModel(base_model, 2)
    print("\n--- 步骤 4.2: 配置 LoRA ---")
    model.base_model = FastLanguageModel.get_peft_model(
        model.base_model,
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj" , "all-linear"],
    )
    print("LoRA 配置完成。")
    print("用哑层替换 lm_head 以节省显存并兼容 Unsloth...")
    original_model = model.base_model.get_base_model()
    hidden_size = original_model.config.hidden_size
  
    # 1. 创建一个新的、极小的线性层作为哑层
    dummy_lm_head = torch.nn.Linear(hidden_size, 1, bias=False)
  
    # 2. 明确地将这个哑层的参数设置为不可训练
    for param in dummy_lm_head.parameters():
        param.requires_grad = False
  
    # 3. 将这个不可训练的哑层赋给模型
    original_model.lm_head = dummy_lm_head
    # print(model)
    print("\n--- 步骤 4.3: 准备训练数据集 ---")
    dataframe = get_dataframe_to_train(DATA_PATH)
    dataframe["text"] = dataframe.apply(create_classifier_input, axis=1)
    dataframe['label'] = dataframe['label'].astype(int)
    dataset = Dataset.from_pandas(dataframe[["text", "label"]])
  
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
    train_dataset = dataset.map(tokenize_function, batched=True)
    train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])
    print(f"数据集准备完毕，总计 {len(train_dataset)} 条训练样本。")
    print("\n--- 步骤 4.4: 设置训练参数 ---")
    training_args = TrainingArguments(
        output_dir=MODEL_OUTPUT_PATH,
        num_train_epochs=2,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        remove_unused_columns=True,
        report_to="none",
        torch_compile=True,
        ddp_find_unused_parameters=False,
        seed=SEED,
        group_by_length=True
    )
    print("\n--- 步骤 4.5: 初始化并开始训练 ---")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()
    print("\n--- 训练完成！ ---")
    print(f"\n--- 步骤 4.6: 保存适配器权重到 {MODEL_OUTPUT_PATH} ---")
    model.base_model.save_pretrained(MODEL_OUTPUT_PATH)
    torch.save(model.classifier.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "classifier.pth"))
    tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
    print("模型已成功保存。")
# ==============================================================================
# 步骤 6: 运行完整的训练和推理流程
# ==============================================================================
if __name__ == "__main__" or "kaggle.python" in str(get_ipython()):
    # 读取测试文件以检查其行数
    test_df_check = pd.read_csv(TEST_FILE_PATH)
  
    # 检查行数是否小于100
    if len(test_df_check) < 100:
        print("测试文件行数小于100，判定为测试运行。")
        print("将直接生成一个空的 submission.csv 文件。")
      
        # 创建一个包含 'row_id' 和 'rule_violation' 列的空 DataFrame
        # 'row_id' 来自于我们读取的测试文件
        submission_df = pd.DataFrame({
            'row_id': test_df_check['row_id'],
            'rule_violation': 0.5 # 通常用0.5作为默认值
        })
      
        # 保存为空的提交文件
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
      
        print(f"空的提交文件 '{SUBMISSION_FILE_PATH}' 已生成。")
        print("文件预览:")
        print(submission_df.head())
      
    else:
        print("测试文件行数大于等于100，开始完整的训练和推理流程。")
      
        # 第一步：训练模型
        run_training()

Writing trainer_script_44.py


In [9]:
%%writefile trainer_script_45.py
# ==============================================================================
# 步骤 1: 导入所有必需的库
# ==============================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1" # 关闭联网，否则unsloth会卡住
import unsloth
from unsloth import FastLanguageModel
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from peft import PeftModel
from tqdm import tqdm
import gc
import torch.nn.functional as F
import random

SEED = 3045

# ==============================================================================
#  ⭐ 新增：设置全局随机种子以保证可复现性 ⭐
# ==============================================================================
def set_seed(seed_value=SEED):
    """设置所有相关的随机种子"""
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # if you are using multi-GPU.

# 调用函数
set_seed()

# ==============================================================================
#  ⭐ 新增：设置CUDA操作为确定性模式 ⭐
# ==============================================================================
# 这会牺牲一些性能，但对于复现性至关重要
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ==============================================================================
# 步骤 2: 定义常量和配置
# ==============================================================================
# 定义输入/输出路径
BASE_MODEL_PATH = "/kaggle/input/qwen3-8b-unsloth-bnb-4bit/transformers/default/1" # 7:45
BASE_MODEL_PATH = "/kaggle/input/llama-3-8b-instruct-bnb-4bit/transformers/default/1/llama-3" # 7:45
BASE_MODEL_PATH = "/kaggle/input/unslothgemma-2-9b-bnb-4bit/tensorflow2/default/1/gemma-2-9b-bnb-4bit" # 12:41
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit" # 17
# BASE_MODEL_PATH = "/kaggle/input/unslothqwen2.5-14b-bnb-4bit/tensorflow2/default/1/Qwen2.5-14B-bnb-4bit" # 暂时无法运行后续找找原因，好像是先加载到GPU:0上再搬到GPU:1上导致00M
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit" # 15
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"
# DATA_PATH = "/kaggle/working/"
TEST_FILE_PATH = os.path.join(DATA_PATH, "test.csv")
# 统一的模型输出和提交文件路径
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_45/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission.csv"
# 确保输出目录存在
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# ==============================================================================
# 3. 定义所有辅助函数
# ==============================================================================
class ClassificationModel(torch.nn.Module): # 参考AutoModelForSequenceClassification的实现，保持一致
    def __init__(self, base_model, num_classes):
        super().__init__()
        self.base_model = base_model
        self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 步骤 1: 调用基础模型获取最后一层的隐藏状态
        # `outputs.hidden_states` 只有在 `output_hidden_states=True` 时才会返回
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        # 步骤 2: 智能定位最后一个有效 token 的索引
        # 获取基础模型（原始的、未被PEFT包装的模型）的 pad_token_id
        # PeftModel 会自动将 config 传递过来
        pad_token_id = self.base_model.config.pad_token_id
      
        # 创建一个掩码，标记出所有非填充 token
        # .ne() 是 "not equal" 的缩写
        non_pad_mask = input_ids.ne(pad_token_id)
      
        # 计算每个序列的实际长度，并找到最后一个有效 token 的索引
        # .sum(dim=1) 计算每行的 True (即 1) 的数量
        # 减 1 是因为索引是从 0 开始的
        last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
      
        # 步骤 3: 使用计算出的索引来提取每个序列对应的隐藏状态
        batch_size = input_ids.shape[0]
        # 使用 torch.gather 沿着 hidden_size 维度精确提取
        # 首先，我们需要将索引张量调整为正确的形状以进行 gather 操作
        # 形状需要是 (batch_size, 1, hidden_size) 以便从 (batch_size, seq_len, hidden_size) 中提取
        expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
      
        # 使用 gather 从 hidden_states 中提取出每个序列最后一个有效 token 的表征
        pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
        # 步骤 4: 将提取出的表征送入分类头
        logits = self.classifier(pooled_hidden_states)
      
        # 步骤 5: 计算损失（如果提供了标签）并返回标准输出
        output = {"logits": logits}
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(logits, labels)
            output["loss"] = loss
          
        return output
# ==============================================================================
# 步骤 3: 定义通用的数据处理函数
# ==============================================================================
def create_classifier_input(row):
    """为分类任务创建一个更明确、更丰富的输入文本"""
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
def get_dataframe_to_train(data_path):
    """
    加载、合并和准备训练数据，包括从测试集中提取的样本。
    对于重复的 (body, rule) 组合，通过投票选择出现次数最多的标签。
    """
    train_dataset = pd.read_csv(os.path.join(data_path, "train.csv"))
    test_dataset = pd.read_csv(os.path.join(data_path, "test.csv"))
    flatten = []
    # 保留原始训练集
    train_subset = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_subset = train_subset.rename(columns={"rule_violation": "label"})
    flatten.append(train_subset)
  
    # 从测试集中提取带标签的样本
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["label"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)
    # 丢弃任何关键字段为空的行
    dataframe = dataframe.dropna(subset=['body', 'rule', 'label'])
  
    # --- 核心修改部分 ---
    # 步骤 1: 计算每个 (body, rule, label) 组合的出现次数
    counts = dataframe.groupby(['body', 'rule', 'label']).size().reset_index(name='counts')
  
    # 步骤 2: 找出每个 (body, rule) 组合中，出现次数最多的那个标签
    # 首先按 (body, rule) 分组，然后找到每组中 'counts' 最大值的索引
    idx = counts.groupby(['body', 'rule'])['counts'].transform(max) == counts['counts']
  
    # 使用该索引筛选出最常见的组合
    # 如果有多个标签出现次数相同，.drop_duplicates会保留第一个
    most_common_df = counts[idx].drop_duplicates(subset=['body', 'rule'], keep='first')
  
    # 步骤 3: 从原始 DataFrame 中筛选出 subreddit 信息并合并
    # 我们需要保留 subreddit 列，但它不在我们的分组键中
    # 先对原始数据去重，以防一个样本有多个 subreddit
    subreddit_info = dataframe[['body', 'rule', 'subreddit']].drop_duplicates(subset=['body', 'rule'])
  
    # 将最常见的标签与 subreddit 信息合并
    final_df = pd.merge(most_common_df[['body', 'rule', 'label']], subreddit_info, on=['body', 'rule'], how='left')

    # 新增: 增加额外的补充数据集
    # 1. 根据train.csv和test.csv中的rule进行去重，获得rules列表
    rules = pd.concat([train_dataset['rule'], test_dataset['rule']]).unique()

    # 2. 获得/kaggle/input/extra-datas/extra.csv的去重后的rule，获得rules_extra列表
    extra_df = pd.read_csv("/kaggle/input/1003-extra-data03/extra_translated_output.csv", encoding='utf-8', encoding_errors='replace')

    rules_extra = extra_df['rule'].unique()

    # 3. 将rules和rules_extra转成小写后进行匹配（仅匹配前5个字符）
    rules_lower = {r.lower()[:5]: r for r in rules}

    # 4. 将extra.csv中的rule_extra通过匹配，得到真正的rule，然后替换
    rule_map = {}
    for extra_r in rules_extra:
        key = extra_r.lower()[:5]
        rule_map[extra_r] = rules_lower[key]
    extra_df['rule'] = extra_df['rule'].map(rule_map)
    extra_df = extra_df.rename(columns={'rule_violation': 'label'})

    # 5. 将extra.csv加入final_df
    final_df = pd.concat([final_df, extra_df[['body', 'rule', 'label', 'subreddit']]], ignore_index=True)

    final_df.to_csv('/kaggle/working/final_df.csv', index=False)
    
    return final_df
# ==============================================================================
# 步骤 4: 定义主训练函数
# ==============================================================================
def run_training():
    """
    执行完整的训练流程。
    """
    print("="*50)
    print(" " * 20 + "开始训练流程")
    print("="*50)
  
    print("\n--- 步骤 4.1: 加载 Tokenizer 和模型 ---")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_PATH,
        max_seq_length=256,
        load_in_4bit=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = ClassificationModel(base_model, 2)
    print("\n--- 步骤 4.2: 配置 LoRA ---")
    model.base_model = FastLanguageModel.get_peft_model(
        model.base_model,
        r=16,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj" , "all-linear"],
    )
    print("LoRA 配置完成。")
    print("用哑层替换 lm_head 以节省显存并兼容 Unsloth...")
    original_model = model.base_model.get_base_model()
    hidden_size = original_model.config.hidden_size
  
    # 1. 创建一个新的、极小的线性层作为哑层
    dummy_lm_head = torch.nn.Linear(hidden_size, 1, bias=False)
  
    # 2. 明确地将这个哑层的参数设置为不可训练
    for param in dummy_lm_head.parameters():
        param.requires_grad = False
  
    # 3. 将这个不可训练的哑层赋给模型
    original_model.lm_head = dummy_lm_head
    # print(model)
    print("\n--- 步骤 4.3: 准备训练数据集 ---")
    dataframe = get_dataframe_to_train(DATA_PATH)
    dataframe["text"] = dataframe.apply(create_classifier_input, axis=1)
    dataframe['label'] = dataframe['label'].astype(int)
    dataset = Dataset.from_pandas(dataframe[["text", "label"]])
  
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
    train_dataset = dataset.map(tokenize_function, batched=True)
    train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in ['input_ids', 'attention_mask', 'label']])
    print(f"数据集准备完毕，总计 {len(train_dataset)} 条训练样本。")
    print("\n--- 步骤 4.4: 设置训练参数 ---")
    training_args = TrainingArguments(
        output_dir=MODEL_OUTPUT_PATH,
        num_train_epochs=2,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=1,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=10,
        save_strategy="no",
        fp16=True,
        remove_unused_columns=True,
        report_to="none",
        torch_compile=True,
        ddp_find_unused_parameters=False,
        seed=SEED,
        group_by_length=True
    )
    print("\n--- 步骤 4.5: 初始化并开始训练 ---")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )
    trainer.train()
    print("\n--- 训练完成！ ---")
    print(f"\n--- 步骤 4.6: 保存适配器权重到 {MODEL_OUTPUT_PATH} ---")
    model.base_model.save_pretrained(MODEL_OUTPUT_PATH)
    torch.save(model.classifier.state_dict(), os.path.join(MODEL_OUTPUT_PATH, "classifier.pth"))
    tokenizer.save_pretrained(MODEL_OUTPUT_PATH)
    print("模型已成功保存。")
# ==============================================================================
# 步骤 6: 运行完整的训练和推理流程
# ==============================================================================
if __name__ == "__main__" or "kaggle.python" in str(get_ipython()):
    # 读取测试文件以检查其行数
    test_df_check = pd.read_csv(TEST_FILE_PATH)
  
    # 检查行数是否小于100
    if len(test_df_check) < 100:
        print("测试文件行数小于100，判定为测试运行。")
        print("将直接生成一个空的 submission.csv 文件。")
      
        # 创建一个包含 'row_id' 和 'rule_violation' 列的空 DataFrame
        # 'row_id' 来自于我们读取的测试文件
        submission_df = pd.DataFrame({
            'row_id': test_df_check['row_id'],
            'rule_violation': 0.5 # 通常用0.5作为默认值
        })
      
        # 保存为空的提交文件
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
      
        print(f"空的提交文件 '{SUBMISSION_FILE_PATH}' 已生成。")
        print("文件预览:")
        print(submission_df.head())
      
    else:
        print("测试文件行数大于等于100，开始完整的训练和推理流程。")
      
        # 第一步：训练模型
        run_training()

Writing trainer_script_45.py


In [10]:
import subprocess
import os
import sys

# 定义两个训练任务的脚本和日志文件
tasks = {
    'GPU 0': {
        'script': 'trainer_script_44.py',
        'log': '/kaggle/working/trainer_script_44.log',
        'env': {'CUDA_VISIBLE_DEVICES': '0'}
    },
    'GPU 1': {
        'script': 'trainer_script_45.py',
        'log': '/kaggle/working/trainer_script_45.log',
        'env': {'CUDA_VISIBLE_DEVICES': '1'}
    }
}

processes = []
log_files = {}

# 循环启动所有训练任务
for device, config in tasks.items():
    # 准备环境，继承当前环境并更新GPU设置
    env = os.environ.copy()
    env.update(config['env'])
    
    # 构建命令
    command = [sys.executable, config['script']]
    
    # 打开日志文件
    log_file = open(config['log'], 'w')
    log_files[device] = log_file
    
    print(f"Starting training for {device} on script {config['script']}...")
    # 使用 Popen 启动非阻塞的后台进程
    p = subprocess.Popen(command, stdout=log_file, stderr=log_file, env=env)
    processes.append(p)

# 等待所有进程完成
print("\nBoth training processes are running in the background. Waiting for them to complete...")
for p in processes:
    p.wait()

# 关闭所有日志文件
for f in log_files.values():
    f.close()

print("\nAll training processes have finished. Check the log files for details:")
for device, config in tasks.items():
    print(f"- {device}: {config['log']}")

Starting training for GPU 0 on script trainer_script_44.py...
Starting training for GPU 1 on script trainer_script_45.py...

Both training processes are running in the background. Waiting for them to complete...

All training processes have finished. Check the log files for details:
- GPU 0: /kaggle/working/trainer_script_44.log
- GPU 1: /kaggle/working/trainer_script_45.log


-----------------------

# 推理42，43, 44, 45

In [11]:
%%writefile predicter_script.py
# 2:47 速度快了一倍！因为是并行两块GPU
import os
# 设置 HF_HUB_OFFLINE
os.environ["HF_HUB_OFFLINE"] = "1"
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch.multiprocessing as mp
import time
from transformers import AutoTokenizer  # 全局加载 tokenizer
from datasets import Dataset
# ==============================================================================
# 全局区域：只保留不依赖 GPU 的库和函数
# ==============================================================================
# 定义常量和配置
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit"
# 【修改】根据你的要求，更改测试文件路径
TEST_FILE_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
# TEST_FILE_PATH = "/kaggle/working/test.csv" # 公平计算推理时间的标准
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_42/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission_42.csv"
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# 辅助函数：创建输入文本
def create_classifier_input(row):
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
# ==============================================================================
# 推理工作函数：一个完全自包含的“黑箱”
# ==============================================================================
def run_inference_on_gpu(gpu_id, data_chunk_df, result_queue):
    """
    此函数在一个指定的GPU上执行所有操作。
    为了保证环境隔离，所有 torch/unsloth/transformers 相关的导入和类定义都在此函数内部。
    """
    # 1. 设置环境变量，必须在导入 torch 之前完成
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["HF_HUB_OFFLINE"] = "1"
    # 2. 在函数内部导入所有必要的、与 GPU 相关的库
    import torch
    import gc
    from unsloth import FastLanguageModel
    from transformers import AutoTokenizer, DataCollatorWithPadding
    from peft import PeftModel
    from datasets import Dataset
    from torch.utils.data import DataLoader
    # 3. 在函数内部定义模型类
    class ClassificationModel(torch.nn.Module):
        def __init__(self, base_model, num_classes):
            super().__init__()
            self.base_model = base_model
            self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
        def forward(self, input_ids, attention_mask=None, labels=None):
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states[-1]
            pad_token_id = self.base_model.config.pad_token_id
            non_pad_mask = input_ids.ne(pad_token_id)
            last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
            expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
            pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
            logits = self.classifier(pooled_hidden_states)
            return {"logits": logits}
    try:
        # 4. 执行标准的单 GPU 推理流程
        print(f"[GPU {gpu_id}]: 环境设置完毕，开始加载模型...")
        device = torch.device("cuda:0") # 对此进程来说，'cuda:0' 就是它唯一能看到的GPU
        tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL_PATH,
            max_seq_length=256,
            load_in_4bit=True,
        )
        model = ClassificationModel(base_model, 2)
        model.base_model = PeftModel.from_pretrained(model.base_model, MODEL_OUTPUT_PATH)
       
        # 【核心修正】: 使用 map_location 将权重加载到当前进程可见的设备上
        classifier_path = os.path.join(MODEL_OUTPUT_PATH, "classifier.pth")
        state_dict = torch.load(classifier_path, map_location=device)
        model.classifier.load_state_dict(state_dict)
       
        model.to(device)
        model.classifier.half()
        model.eval()
        # 只编译分类头！
        model.classifier = torch.compile(model.classifier, mode="reduce-overhead", fullgraph=True)
       
        print(f"[GPU {gpu_id}]: 模型加载完成。")
        # 数据已预 token化，直接使用
        tokenized_test_dataset = Dataset.from_pandas(data_chunk_df[['input_ids', 'attention_mask']].reset_index(drop=True))
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        inference_dataloader = DataLoader(
            tokenized_test_dataset,
            batch_size=16,
            collate_fn=data_collator
        )
        all_positive_probabilities = []
        with torch.no_grad():
            progress_bar = tqdm(inference_dataloader, desc=f"Predicting on GPU {gpu_id}", position=gpu_id)
            for batch in progress_bar:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                logits_cpu = outputs["logits"].cpu()
                probabilities = torch.softmax(logits_cpu, dim=-1)
                positive_probabilities = probabilities[:, 1].numpy()
                all_positive_probabilities.extend(positive_probabilities)
        results = {index: prob for index, prob in zip(data_chunk_df['original_index'], all_positive_probabilities)}
        result_queue.put(results)
        print(f"[GPU {gpu_id}]: 推理完成。")
    except Exception as e:
        print(f"Error on GPU {gpu_id}: {e}")
        import traceback
        traceback.print_exc()
        result_queue.put({})
    finally:
        del model, base_model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
# ==============================================================================
# 主协调函数
# ==============================================================================
def run_parallel_inference_and_create_submission():
    print("\n" + "="*50)
    print(" " * 10 + "开始手动并行推理流程 (V4 - 最终版)")
    print("="*50)
    print("\n--- 步骤 1: 加载并准备测试数据 ---")
    # 确保测试文件存在，如果不存在，创建一个假的用于测试
    if not os.path.exists(TEST_FILE_PATH):
        print(f"警告：测试文件 {TEST_FILE_PATH} 不存在。将创建一个假的测试文件。")
        pd.DataFrame({
            'row_id': range(200),
            'body': ['test body'] * 200,
            'rule': ['test rule'] * 200
        }).to_csv(TEST_FILE_PATH, index=False)
       
    test_df = pd.read_csv(TEST_FILE_PATH)
    test_df["text"] = test_df.apply(create_classifier_input, axis=1)
   
    unique_df = test_df.drop_duplicates(subset=['body', 'rule']).copy()
    print(f"原始测试样本数: {len(test_df)}, 唯一 body-rule 组合数: {len(unique_df)}")
   
    if len(unique_df) == 0:
        print("没有可推理的唯一数据。")
        submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': 0.5})
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        print("已生成占位符提交文件。")
        return
   
    # 加载 tokenizer（全局）
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
   
    # 准备数据集
    unique_df['original_index'] = unique_df.index
    test_dataset = Dataset.from_pandas(unique_df[['original_index', 'text', 'body', 'rule']])
   
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
   
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, num_proc=4)
   
    # 添加长度并排序
    def add_length(examples):
        examples['length'] = [len(ids) for ids in examples['input_ids']]
        return examples
   
    tokenized_test_dataset = tokenized_test_dataset.map(add_length, batched=True, batch_size=1000)
    tokenized_test_dataset = tokenized_test_dataset.sort('length')
   
    # 分割已排序的 tokenized 数据（转换为 pd 以兼容原代码）
    tokenized_df = tokenized_test_dataset.to_pandas()
    num_gpus = 2
    # 在主函数中，排序后添加以下函数
    def balanced_split(df, num_splits=2):
        # 按 length 排序已完成
        total_length = df['length'].sum()
        target_per_split = total_length / num_splits
        chunks = [[] for _ in range(num_splits)]
        current_sums = [0] * num_splits
        for _, row in df.iterrows():
            # 分配到当前总和最小的 chunk
            min_idx = current_sums.index(min(current_sums))
            chunks[min_idx].append(row)
            current_sums[min_idx] += row['length']
        # 转换为 DataFrame
        data_chunks = [pd.DataFrame(chunk) for chunk in chunks]
        return data_chunks
    
    # 替换原分割
    # data_chunks = np.array_split(tokenized_df, num_gpus)
    data_chunks = balanced_split(tokenized_df, num_gpus)
    print(f"数据被分割成 {len(data_chunks)} 块，每块大小约为 {len(data_chunks[0])}")
   
    mp.set_start_method('spawn', force=True)
    result_queue = mp.Queue()
    processes = []
    for i in range(num_gpus):
        if len(data_chunks[i]) > 0:
            p = mp.Process(target=run_inference_on_gpu, args=(i, data_chunks[i], result_queue))
            processes.append(p)
            p.start()
            if i < num_gpus - 1: # Wait before starting the next process
                print(f"Waiting 5 seconds to start the next process to avoid initialization conflicts...")
                time.sleep(5)
    all_results = {}
    for _ in range(len(processes)):
        results_dict = result_queue.get()
        if results_dict:
            all_results.update(results_dict)
    for p in processes:
        p.join()
    if len(all_results) != len(unique_df):
        print(f"警告：部分进程推理失败！成功处理 {len(all_results)} 条，预期 {len(unique_df)} 条。结果可能不完整！")
   
    print("\n--- 步骤 2: 合并结果并生成提交文件 ---")
   
    result_series = pd.Series(all_results, name='rule_violation_pred')
    unique_df = unique_df.set_index('original_index').join(result_series).reset_index(drop=True)
   
    submission_df = test_df.merge(unique_df[['body', 'rule', 'rule_violation_pred']], on=['body', 'rule'], how='left')
    submission_df = submission_df.rename(columns={'rule_violation_pred': 'rule_violation'})
    # 为可能失败的行填充默认值
    submission_df['rule_violation'] = submission_df['rule_violation'].fillna(0.5)
    submission_df = submission_df[['row_id', 'rule_violation']]
   
    assert len(submission_df) == len(test_df), "Submission DataFrame 行数与原始测试数据不匹配！"
   
    submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
   
    print("\n========================================================")
    print(f"提交文件 '{SUBMISSION_FILE_PATH}' 已成功生成！")
    print("文件预览:")
    print(submission_df.head())
    print("========================================================")
# ==============================================================================
# 运行主流程
# ==============================================================================
if __name__ == "__main__":
    try:
        test_df_check = pd.read_csv(TEST_FILE_PATH)
        if len(test_df_check) < 100:
            print("测试文件为空，生成占位符提交文件。")
            submission_df = pd.DataFrame({'row_id': test_df_check['row_id'], 'rule_violation': 0.5})
            submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        else:
            run_parallel_inference_and_create_submission()
    except FileNotFoundError:
        print(f"错误: 测试文件 '{TEST_FILE_PATH}' 未找到。请确保该文件存在。")
        # 创建一个空的提交文件以避免 Kaggle 报错
        pd.DataFrame({'row_id': [], 'rule_violation': []}).to_csv(SUBMISSION_FILE_PATH, index=False)

Writing predicter_script.py


In [12]:
!python predicter_script.py

测试文件为空，生成占位符提交文件。


In [13]:
%%writefile predicter_script.py
# 2:47 速度快了一倍！因为是并行两块GPU
import os
# 设置 HF_HUB_OFFLINE
os.environ["HF_HUB_OFFLINE"] = "1"
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch.multiprocessing as mp
import time
from transformers import AutoTokenizer  # 全局加载 tokenizer
from datasets import Dataset
# ==============================================================================
# 全局区域：只保留不依赖 GPU 的库和函数
# ==============================================================================
# 定义常量和配置
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit"
# 【修改】根据你的要求，更改测试文件路径
TEST_FILE_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
# TEST_FILE_PATH = "/kaggle/working/test.csv" # 公平计算推理时间的标准
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_43/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission_43.csv"
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# 辅助函数：创建输入文本
def create_classifier_input(row):
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
# ==============================================================================
# 推理工作函数：一个完全自包含的“黑箱”
# ==============================================================================
def run_inference_on_gpu(gpu_id, data_chunk_df, result_queue):
    """
    此函数在一个指定的GPU上执行所有操作。
    为了保证环境隔离，所有 torch/unsloth/transformers 相关的导入和类定义都在此函数内部。
    """
    # 1. 设置环境变量，必须在导入 torch 之前完成
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["HF_HUB_OFFLINE"] = "1"
    # 2. 在函数内部导入所有必要的、与 GPU 相关的库
    import torch
    import gc
    from unsloth import FastLanguageModel
    from transformers import AutoTokenizer, DataCollatorWithPadding
    from peft import PeftModel
    from datasets import Dataset
    from torch.utils.data import DataLoader
    # 3. 在函数内部定义模型类
    class ClassificationModel(torch.nn.Module):
        def __init__(self, base_model, num_classes):
            super().__init__()
            self.base_model = base_model
            self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
        def forward(self, input_ids, attention_mask=None, labels=None):
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states[-1]
            pad_token_id = self.base_model.config.pad_token_id
            non_pad_mask = input_ids.ne(pad_token_id)
            last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
            expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
            pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
            logits = self.classifier(pooled_hidden_states)
            return {"logits": logits}
    try:
        # 4. 执行标准的单 GPU 推理流程
        print(f"[GPU {gpu_id}]: 环境设置完毕，开始加载模型...")
        device = torch.device("cuda:0") # 对此进程来说，'cuda:0' 就是它唯一能看到的GPU
        tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL_PATH,
            max_seq_length=256,
            load_in_4bit=True,
        )
        model = ClassificationModel(base_model, 2)
        model.base_model = PeftModel.from_pretrained(model.base_model, MODEL_OUTPUT_PATH)
       
        # 【核心修正】: 使用 map_location 将权重加载到当前进程可见的设备上
        classifier_path = os.path.join(MODEL_OUTPUT_PATH, "classifier.pth")
        state_dict = torch.load(classifier_path, map_location=device)
        model.classifier.load_state_dict(state_dict)
       
        model.to(device)
        model.classifier.half()
        model.eval()
        # 只编译分类头！
        model.classifier = torch.compile(model.classifier, mode="reduce-overhead", fullgraph=True)
       
        print(f"[GPU {gpu_id}]: 模型加载完成。")
        # 数据已预 token化，直接使用
        tokenized_test_dataset = Dataset.from_pandas(data_chunk_df[['input_ids', 'attention_mask']].reset_index(drop=True))
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        inference_dataloader = DataLoader(
            tokenized_test_dataset,
            batch_size=16,
            collate_fn=data_collator
        )
        all_positive_probabilities = []
        with torch.no_grad():
            progress_bar = tqdm(inference_dataloader, desc=f"Predicting on GPU {gpu_id}", position=gpu_id)
            for batch in progress_bar:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                logits_cpu = outputs["logits"].cpu()
                probabilities = torch.softmax(logits_cpu, dim=-1)
                positive_probabilities = probabilities[:, 1].numpy()
                all_positive_probabilities.extend(positive_probabilities)
        results = {index: prob for index, prob in zip(data_chunk_df['original_index'], all_positive_probabilities)}
        result_queue.put(results)
        print(f"[GPU {gpu_id}]: 推理完成。")
    except Exception as e:
        print(f"Error on GPU {gpu_id}: {e}")
        import traceback
        traceback.print_exc()
        result_queue.put({})
    finally:
        del model, base_model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
# ==============================================================================
# 主协调函数
# ==============================================================================
def run_parallel_inference_and_create_submission():
    print("\n" + "="*50)
    print(" " * 10 + "开始手动并行推理流程 (V4 - 最终版)")
    print("="*50)
    print("\n--- 步骤 1: 加载并准备测试数据 ---")
    # 确保测试文件存在，如果不存在，创建一个假的用于测试
    if not os.path.exists(TEST_FILE_PATH):
        print(f"警告：测试文件 {TEST_FILE_PATH} 不存在。将创建一个假的测试文件。")
        pd.DataFrame({
            'row_id': range(200),
            'body': ['test body'] * 200,
            'rule': ['test rule'] * 200
        }).to_csv(TEST_FILE_PATH, index=False)
       
    test_df = pd.read_csv(TEST_FILE_PATH)
    test_df["text"] = test_df.apply(create_classifier_input, axis=1)
   
    unique_df = test_df.drop_duplicates(subset=['body', 'rule']).copy()
    print(f"原始测试样本数: {len(test_df)}, 唯一 body-rule 组合数: {len(unique_df)}")
   
    if len(unique_df) == 0:
        print("没有可推理的唯一数据。")
        submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': 0.5})
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        print("已生成占位符提交文件。")
        return
   
    # 加载 tokenizer（全局）
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
   
    # 准备数据集
    unique_df['original_index'] = unique_df.index
    test_dataset = Dataset.from_pandas(unique_df[['original_index', 'text', 'body', 'rule']])
   
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
   
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, num_proc=4)
   
    # 添加长度并排序
    def add_length(examples):
        examples['length'] = [len(ids) for ids in examples['input_ids']]
        return examples
   
    tokenized_test_dataset = tokenized_test_dataset.map(add_length, batched=True, batch_size=1000)
    tokenized_test_dataset = tokenized_test_dataset.sort('length')
   
    # 分割已排序的 tokenized 数据（转换为 pd 以兼容原代码）
    tokenized_df = tokenized_test_dataset.to_pandas()
    num_gpus = 2
    # 在主函数中，排序后添加以下函数
    def balanced_split(df, num_splits=2):
        # 按 length 排序已完成
        total_length = df['length'].sum()
        target_per_split = total_length / num_splits
        chunks = [[] for _ in range(num_splits)]
        current_sums = [0] * num_splits
        for _, row in df.iterrows():
            # 分配到当前总和最小的 chunk
            min_idx = current_sums.index(min(current_sums))
            chunks[min_idx].append(row)
            current_sums[min_idx] += row['length']
        # 转换为 DataFrame
        data_chunks = [pd.DataFrame(chunk) for chunk in chunks]
        return data_chunks
    
    # 替换原分割
    # data_chunks = np.array_split(tokenized_df, num_gpus)
    data_chunks = balanced_split(tokenized_df, num_gpus)
    print(f"数据被分割成 {len(data_chunks)} 块，每块大小约为 {len(data_chunks[0])}")
   
    mp.set_start_method('spawn', force=True)
    result_queue = mp.Queue()
    processes = []
    for i in range(num_gpus):
        if len(data_chunks[i]) > 0:
            p = mp.Process(target=run_inference_on_gpu, args=(i, data_chunks[i], result_queue))
            processes.append(p)
            p.start()
            if i < num_gpus - 1: # Wait before starting the next process
                print(f"Waiting 5 seconds to start the next process to avoid initialization conflicts...")
                time.sleep(5)
    all_results = {}
    for _ in range(len(processes)):
        results_dict = result_queue.get()
        if results_dict:
            all_results.update(results_dict)
    for p in processes:
        p.join()
    if len(all_results) != len(unique_df):
        print(f"警告：部分进程推理失败！成功处理 {len(all_results)} 条，预期 {len(unique_df)} 条。结果可能不完整！")
   
    print("\n--- 步骤 2: 合并结果并生成提交文件 ---")
   
    result_series = pd.Series(all_results, name='rule_violation_pred')
    unique_df = unique_df.set_index('original_index').join(result_series).reset_index(drop=True)
   
    submission_df = test_df.merge(unique_df[['body', 'rule', 'rule_violation_pred']], on=['body', 'rule'], how='left')
    submission_df = submission_df.rename(columns={'rule_violation_pred': 'rule_violation'})
    # 为可能失败的行填充默认值
    submission_df['rule_violation'] = submission_df['rule_violation'].fillna(0.5)
    submission_df = submission_df[['row_id', 'rule_violation']]
   
    assert len(submission_df) == len(test_df), "Submission DataFrame 行数与原始测试数据不匹配！"
   
    submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
   
    print("\n========================================================")
    print(f"提交文件 '{SUBMISSION_FILE_PATH}' 已成功生成！")
    print("文件预览:")
    print(submission_df.head())
    print("========================================================")
# ==============================================================================
# 运行主流程
# ==============================================================================
if __name__ == "__main__":
    try:
        test_df_check = pd.read_csv(TEST_FILE_PATH)
        if len(test_df_check) < 100:
            print("测试文件为空，生成占位符提交文件。")
            submission_df = pd.DataFrame({'row_id': test_df_check['row_id'], 'rule_violation': 0.5})
            submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        else:
            run_parallel_inference_and_create_submission()
    except FileNotFoundError:
        print(f"错误: 测试文件 '{TEST_FILE_PATH}' 未找到。请确保该文件存在。")
        # 创建一个空的提交文件以避免 Kaggle 报错
        pd.DataFrame({'row_id': [], 'rule_violation': []}).to_csv(SUBMISSION_FILE_PATH, index=False)

Overwriting predicter_script.py


In [14]:
!python predicter_script.py

测试文件为空，生成占位符提交文件。


In [15]:
%%writefile predicter_script.py
# 2:47 速度快了一倍！因为是并行两块GPU
import os
# 设置 HF_HUB_OFFLINE
os.environ["HF_HUB_OFFLINE"] = "1"
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch.multiprocessing as mp
import time
from transformers import AutoTokenizer  # 全局加载 tokenizer
from datasets import Dataset
# ==============================================================================
# 全局区域：只保留不依赖 GPU 的库和函数
# ==============================================================================
# 定义常量和配置
BASE_MODEL_PATH = "/kaggle/input/0910-unslothqwen3-14b-unsloth-bnb-4bit/tensorflow2/default/1/Qwen3-14B-unsloth-bnb-4bit"
# 【修改】根据你的要求，更改测试文件路径
TEST_FILE_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
# TEST_FILE_PATH = "/kaggle/working/test.csv" # 公平计算推理时间的标准
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_44/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission_44.csv"
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# 辅助函数：创建输入文本
def create_classifier_input(row):
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
# ==============================================================================
# 推理工作函数：一个完全自包含的“黑箱”
# ==============================================================================
def run_inference_on_gpu(gpu_id, data_chunk_df, result_queue):
    """
    此函数在一个指定的GPU上执行所有操作。
    为了保证环境隔离，所有 torch/unsloth/transformers 相关的导入和类定义都在此函数内部。
    """
    # 1. 设置环境变量，必须在导入 torch 之前完成
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["HF_HUB_OFFLINE"] = "1"
    # 2. 在函数内部导入所有必要的、与 GPU 相关的库
    import torch
    import gc
    from unsloth import FastLanguageModel
    from transformers import AutoTokenizer, DataCollatorWithPadding
    from peft import PeftModel
    from datasets import Dataset
    from torch.utils.data import DataLoader
    # 3. 在函数内部定义模型类
    class ClassificationModel(torch.nn.Module):
        def __init__(self, base_model, num_classes):
            super().__init__()
            self.base_model = base_model
            self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
        def forward(self, input_ids, attention_mask=None, labels=None):
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states[-1]
            pad_token_id = self.base_model.config.pad_token_id
            non_pad_mask = input_ids.ne(pad_token_id)
            last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
            expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
            pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
            logits = self.classifier(pooled_hidden_states)
            return {"logits": logits}
    try:
        # 4. 执行标准的单 GPU 推理流程
        print(f"[GPU {gpu_id}]: 环境设置完毕，开始加载模型...")
        device = torch.device("cuda:0") # 对此进程来说，'cuda:0' 就是它唯一能看到的GPU
        tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL_PATH,
            max_seq_length=256,
            load_in_4bit=True,
        )
        model = ClassificationModel(base_model, 2)
        model.base_model = PeftModel.from_pretrained(model.base_model, MODEL_OUTPUT_PATH)
       
        # 【核心修正】: 使用 map_location 将权重加载到当前进程可见的设备上
        classifier_path = os.path.join(MODEL_OUTPUT_PATH, "classifier.pth")
        state_dict = torch.load(classifier_path, map_location=device)
        model.classifier.load_state_dict(state_dict)
       
        model.to(device)
        model.classifier.half()
        model.eval()
        # 只编译分类头！
        model.classifier = torch.compile(model.classifier, mode="reduce-overhead", fullgraph=True)
       
        print(f"[GPU {gpu_id}]: 模型加载完成。")
        # 数据已预 token化，直接使用
        tokenized_test_dataset = Dataset.from_pandas(data_chunk_df[['input_ids', 'attention_mask']].reset_index(drop=True))
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        inference_dataloader = DataLoader(
            tokenized_test_dataset,
            batch_size=16,
            collate_fn=data_collator
        )
        all_positive_probabilities = []
        with torch.no_grad():
            progress_bar = tqdm(inference_dataloader, desc=f"Predicting on GPU {gpu_id}", position=gpu_id)
            for batch in progress_bar:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                logits_cpu = outputs["logits"].cpu()
                probabilities = torch.softmax(logits_cpu, dim=-1)
                positive_probabilities = probabilities[:, 1].numpy()
                all_positive_probabilities.extend(positive_probabilities)
        results = {index: prob for index, prob in zip(data_chunk_df['original_index'], all_positive_probabilities)}
        result_queue.put(results)
        print(f"[GPU {gpu_id}]: 推理完成。")
    except Exception as e:
        print(f"Error on GPU {gpu_id}: {e}")
        import traceback
        traceback.print_exc()
        result_queue.put({})
    finally:
        del model, base_model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
# ==============================================================================
# 主协调函数
# ==============================================================================
def run_parallel_inference_and_create_submission():
    print("\n" + "="*50)
    print(" " * 10 + "开始手动并行推理流程 (V4 - 最终版)")
    print("="*50)
    print("\n--- 步骤 1: 加载并准备测试数据 ---")
    # 确保测试文件存在，如果不存在，创建一个假的用于测试
    if not os.path.exists(TEST_FILE_PATH):
        print(f"警告：测试文件 {TEST_FILE_PATH} 不存在。将创建一个假的测试文件。")
        pd.DataFrame({
            'row_id': range(200),
            'body': ['test body'] * 200,
            'rule': ['test rule'] * 200
        }).to_csv(TEST_FILE_PATH, index=False)
       
    test_df = pd.read_csv(TEST_FILE_PATH)
    test_df["text"] = test_df.apply(create_classifier_input, axis=1)
   
    unique_df = test_df.drop_duplicates(subset=['body', 'rule']).copy()
    print(f"原始测试样本数: {len(test_df)}, 唯一 body-rule 组合数: {len(unique_df)}")
   
    if len(unique_df) == 0:
        print("没有可推理的唯一数据。")
        submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': 0.5})
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        print("已生成占位符提交文件。")
        return
   
    # 加载 tokenizer（全局）
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
   
    # 准备数据集
    unique_df['original_index'] = unique_df.index
    test_dataset = Dataset.from_pandas(unique_df[['original_index', 'text', 'body', 'rule']])
   
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
   
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, num_proc=4)
   
    # 添加长度并排序
    def add_length(examples):
        examples['length'] = [len(ids) for ids in examples['input_ids']]
        return examples
   
    tokenized_test_dataset = tokenized_test_dataset.map(add_length, batched=True, batch_size=1000)
    tokenized_test_dataset = tokenized_test_dataset.sort('length')
   
    # 分割已排序的 tokenized 数据（转换为 pd 以兼容原代码）
    tokenized_df = tokenized_test_dataset.to_pandas()
    num_gpus = 2
    # 在主函数中，排序后添加以下函数
    def balanced_split(df, num_splits=2):
        # 按 length 排序已完成
        total_length = df['length'].sum()
        target_per_split = total_length / num_splits
        chunks = [[] for _ in range(num_splits)]
        current_sums = [0] * num_splits
        for _, row in df.iterrows():
            # 分配到当前总和最小的 chunk
            min_idx = current_sums.index(min(current_sums))
            chunks[min_idx].append(row)
            current_sums[min_idx] += row['length']
        # 转换为 DataFrame
        data_chunks = [pd.DataFrame(chunk) for chunk in chunks]
        return data_chunks
    
    # 替换原分割
    # data_chunks = np.array_split(tokenized_df, num_gpus)
    data_chunks = balanced_split(tokenized_df, num_gpus)
    print(f"数据被分割成 {len(data_chunks)} 块，每块大小约为 {len(data_chunks[0])}")
   
    mp.set_start_method('spawn', force=True)
    result_queue = mp.Queue()
    processes = []
    for i in range(num_gpus):
        if len(data_chunks[i]) > 0:
            p = mp.Process(target=run_inference_on_gpu, args=(i, data_chunks[i], result_queue))
            processes.append(p)
            p.start()
            if i < num_gpus - 1: # Wait before starting the next process
                print(f"Waiting 5 seconds to start the next process to avoid initialization conflicts...")
                time.sleep(5)
    all_results = {}
    for _ in range(len(processes)):
        results_dict = result_queue.get()
        if results_dict:
            all_results.update(results_dict)
    for p in processes:
        p.join()
    if len(all_results) != len(unique_df):
        print(f"警告：部分进程推理失败！成功处理 {len(all_results)} 条，预期 {len(unique_df)} 条。结果可能不完整！")
   
    print("\n--- 步骤 2: 合并结果并生成提交文件 ---")
   
    result_series = pd.Series(all_results, name='rule_violation_pred')
    unique_df = unique_df.set_index('original_index').join(result_series).reset_index(drop=True)
   
    submission_df = test_df.merge(unique_df[['body', 'rule', 'rule_violation_pred']], on=['body', 'rule'], how='left')
    submission_df = submission_df.rename(columns={'rule_violation_pred': 'rule_violation'})
    # 为可能失败的行填充默认值
    submission_df['rule_violation'] = submission_df['rule_violation'].fillna(0.5)
    submission_df = submission_df[['row_id', 'rule_violation']]
   
    assert len(submission_df) == len(test_df), "Submission DataFrame 行数与原始测试数据不匹配！"
   
    submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
   
    print("\n========================================================")
    print(f"提交文件 '{SUBMISSION_FILE_PATH}' 已成功生成！")
    print("文件预览:")
    print(submission_df.head())
    print("========================================================")
# ==============================================================================
# 运行主流程
# ==============================================================================
if __name__ == "__main__":
    try:
        test_df_check = pd.read_csv(TEST_FILE_PATH)
        if len(test_df_check) < 100:
            print("测试文件为空，生成占位符提交文件。")
            submission_df = pd.DataFrame({'row_id': test_df_check['row_id'], 'rule_violation': 0.5})
            submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        else:
            run_parallel_inference_and_create_submission()
    except FileNotFoundError:
        print(f"错误: 测试文件 '{TEST_FILE_PATH}' 未找到。请确保该文件存在。")
        # 创建一个空的提交文件以避免 Kaggle 报错
        pd.DataFrame({'row_id': [], 'rule_violation': []}).to_csv(SUBMISSION_FILE_PATH, index=False)

Overwriting predicter_script.py


In [16]:
!python predicter_script.py

测试文件为空，生成占位符提交文件。


In [17]:
%%writefile predicter_script.py
# 2:47 速度快了一倍！因为是并行两块GPU
import os
# 设置 HF_HUB_OFFLINE
os.environ["HF_HUB_OFFLINE"] = "1"
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch.multiprocessing as mp
import time
from transformers import AutoTokenizer  # 全局加载 tokenizer
from datasets import Dataset
# ==============================================================================
# 全局区域：只保留不依赖 GPU 的库和函数
# ==============================================================================
# 定义常量和配置
BASE_MODEL_PATH = "/kaggle/input/unslothllama-2-13b-bnb-4bit/tensorflow2/default/1/llama-2-13b-bnb-4bit"
# 【修改】根据你的要求，更改测试文件路径
TEST_FILE_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
# TEST_FILE_PATH = "/kaggle/working/test.csv" # 公平计算推理时间的标准
MODEL_OUTPUT_PATH = "/kaggle/working/qwen_output_45/"
SUBMISSION_FILE_PATH = "/kaggle/working/submission_45.csv"
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
# 辅助函数：创建输入文本
def create_classifier_input(row):
    prompt_template = f"""You are a Reddit comment moderator. Based on the rule, does the comment post a violation?

Rule: {row['rule']}

Comment: {row['body']}
"""
    return prompt_template
# ==============================================================================
# 推理工作函数：一个完全自包含的“黑箱”
# ==============================================================================
def run_inference_on_gpu(gpu_id, data_chunk_df, result_queue):
    """
    此函数在一个指定的GPU上执行所有操作。
    为了保证环境隔离，所有 torch/unsloth/transformers 相关的导入和类定义都在此函数内部。
    """
    # 1. 设置环境变量，必须在导入 torch 之前完成
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    os.environ["HF_HUB_OFFLINE"] = "1"
    # 2. 在函数内部导入所有必要的、与 GPU 相关的库
    import torch
    import gc
    from unsloth import FastLanguageModel
    from transformers import AutoTokenizer, DataCollatorWithPadding
    from peft import PeftModel
    from datasets import Dataset
    from torch.utils.data import DataLoader
    # 3. 在函数内部定义模型类
    class ClassificationModel(torch.nn.Module):
        def __init__(self, base_model, num_classes):
            super().__init__()
            self.base_model = base_model
            self.classifier = torch.nn.Linear(base_model.config.hidden_size, num_classes, bias=False)
        def forward(self, input_ids, attention_mask=None, labels=None):
            outputs = self.base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden_states = outputs.hidden_states[-1]
            pad_token_id = self.base_model.config.pad_token_id
            non_pad_mask = input_ids.ne(pad_token_id)
            last_non_pad_token_indices = non_pad_mask.sum(dim=1) - 1
            expanded_indices = last_non_pad_token_indices.view(-1, 1, 1).expand(-1, -1, hidden_states.shape[-1])
            pooled_hidden_states = hidden_states.gather(1, expanded_indices).squeeze(1)
            logits = self.classifier(pooled_hidden_states)
            return {"logits": logits}
    try:
        # 4. 执行标准的单 GPU 推理流程
        print(f"[GPU {gpu_id}]: 环境设置完毕，开始加载模型...")
        device = torch.device("cuda:0") # 对此进程来说，'cuda:0' 就是它唯一能看到的GPU
        tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model, _ = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL_PATH,
            max_seq_length=256,
            load_in_4bit=True,
        )
        model = ClassificationModel(base_model, 2)
        model.base_model = PeftModel.from_pretrained(model.base_model, MODEL_OUTPUT_PATH)
       
        # 【核心修正】: 使用 map_location 将权重加载到当前进程可见的设备上
        classifier_path = os.path.join(MODEL_OUTPUT_PATH, "classifier.pth")
        state_dict = torch.load(classifier_path, map_location=device)
        model.classifier.load_state_dict(state_dict)
       
        model.to(device)
        model.classifier.half()
        model.eval()
        # 只编译分类头！
        model.classifier = torch.compile(model.classifier, mode="reduce-overhead", fullgraph=True)
       
        print(f"[GPU {gpu_id}]: 模型加载完成。")
        # 数据已预 token化，直接使用
        tokenized_test_dataset = Dataset.from_pandas(data_chunk_df[['input_ids', 'attention_mask']].reset_index(drop=True))
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        inference_dataloader = DataLoader(
            tokenized_test_dataset,
            batch_size=16,
            collate_fn=data_collator
        )
        all_positive_probabilities = []
        with torch.no_grad():
            progress_bar = tqdm(inference_dataloader, desc=f"Predicting on GPU {gpu_id}", position=gpu_id)
            for batch in progress_bar:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                logits_cpu = outputs["logits"].cpu()
                probabilities = torch.softmax(logits_cpu, dim=-1)
                positive_probabilities = probabilities[:, 1].numpy()
                all_positive_probabilities.extend(positive_probabilities)
        results = {index: prob for index, prob in zip(data_chunk_df['original_index'], all_positive_probabilities)}
        result_queue.put(results)
        print(f"[GPU {gpu_id}]: 推理完成。")
    except Exception as e:
        print(f"Error on GPU {gpu_id}: {e}")
        import traceback
        traceback.print_exc()
        result_queue.put({})
    finally:
        del model, base_model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
# ==============================================================================
# 主协调函数
# ==============================================================================
def run_parallel_inference_and_create_submission():
    print("\n" + "="*50)
    print(" " * 10 + "开始手动并行推理流程 (V4 - 最终版)")
    print("="*50)
    print("\n--- 步骤 1: 加载并准备测试数据 ---")
    # 确保测试文件存在，如果不存在，创建一个假的用于测试
    if not os.path.exists(TEST_FILE_PATH):
        print(f"警告：测试文件 {TEST_FILE_PATH} 不存在。将创建一个假的测试文件。")
        pd.DataFrame({
            'row_id': range(200),
            'body': ['test body'] * 200,
            'rule': ['test rule'] * 200
        }).to_csv(TEST_FILE_PATH, index=False)
       
    test_df = pd.read_csv(TEST_FILE_PATH)
    test_df["text"] = test_df.apply(create_classifier_input, axis=1)
   
    unique_df = test_df.drop_duplicates(subset=['body', 'rule']).copy()
    print(f"原始测试样本数: {len(test_df)}, 唯一 body-rule 组合数: {len(unique_df)}")
   
    if len(unique_df) == 0:
        print("没有可推理的唯一数据。")
        submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': 0.5})
        submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        print("已生成占位符提交文件。")
        return
   
    # 加载 tokenizer（全局）
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
   
    # 准备数据集
    unique_df['original_index'] = unique_df.index
    test_dataset = Dataset.from_pandas(unique_df[['original_index', 'text', 'body', 'rule']])
   
    # Tokenize
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=256)
   
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, num_proc=4)
   
    # 添加长度并排序
    def add_length(examples):
        examples['length'] = [len(ids) for ids in examples['input_ids']]
        return examples
   
    tokenized_test_dataset = tokenized_test_dataset.map(add_length, batched=True, batch_size=1000)
    tokenized_test_dataset = tokenized_test_dataset.sort('length')
   
    # 分割已排序的 tokenized 数据（转换为 pd 以兼容原代码）
    tokenized_df = tokenized_test_dataset.to_pandas()
    num_gpus = 2
    # 在主函数中，排序后添加以下函数
    def balanced_split(df, num_splits=2):
        # 按 length 排序已完成
        total_length = df['length'].sum()
        target_per_split = total_length / num_splits
        chunks = [[] for _ in range(num_splits)]
        current_sums = [0] * num_splits
        for _, row in df.iterrows():
            # 分配到当前总和最小的 chunk
            min_idx = current_sums.index(min(current_sums))
            chunks[min_idx].append(row)
            current_sums[min_idx] += row['length']
        # 转换为 DataFrame
        data_chunks = [pd.DataFrame(chunk) for chunk in chunks]
        return data_chunks
    
    # 替换原分割
    # data_chunks = np.array_split(tokenized_df, num_gpus)
    data_chunks = balanced_split(tokenized_df, num_gpus)
    print(f"数据被分割成 {len(data_chunks)} 块，每块大小约为 {len(data_chunks[0])}")
   
    mp.set_start_method('spawn', force=True)
    result_queue = mp.Queue()
    processes = []
    for i in range(num_gpus):
        if len(data_chunks[i]) > 0:
            p = mp.Process(target=run_inference_on_gpu, args=(i, data_chunks[i], result_queue))
            processes.append(p)
            p.start()
            if i < num_gpus - 1: # Wait before starting the next process
                print(f"Waiting 5 seconds to start the next process to avoid initialization conflicts...")
                time.sleep(5)
    all_results = {}
    for _ in range(len(processes)):
        results_dict = result_queue.get()
        if results_dict:
            all_results.update(results_dict)
    for p in processes:
        p.join()
    if len(all_results) != len(unique_df):
        print(f"警告：部分进程推理失败！成功处理 {len(all_results)} 条，预期 {len(unique_df)} 条。结果可能不完整！")
   
    print("\n--- 步骤 2: 合并结果并生成提交文件 ---")
   
    result_series = pd.Series(all_results, name='rule_violation_pred')
    unique_df = unique_df.set_index('original_index').join(result_series).reset_index(drop=True)
   
    submission_df = test_df.merge(unique_df[['body', 'rule', 'rule_violation_pred']], on=['body', 'rule'], how='left')
    submission_df = submission_df.rename(columns={'rule_violation_pred': 'rule_violation'})
    # 为可能失败的行填充默认值
    submission_df['rule_violation'] = submission_df['rule_violation'].fillna(0.5)
    submission_df = submission_df[['row_id', 'rule_violation']]
   
    assert len(submission_df) == len(test_df), "Submission DataFrame 行数与原始测试数据不匹配！"
   
    submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
   
    print("\n========================================================")
    print(f"提交文件 '{SUBMISSION_FILE_PATH}' 已成功生成！")
    print("文件预览:")
    print(submission_df.head())
    print("========================================================")
# ==============================================================================
# 运行主流程
# ==============================================================================
if __name__ == "__main__":
    try:
        test_df_check = pd.read_csv(TEST_FILE_PATH)
        if len(test_df_check) < 100:
            print("测试文件为空，生成占位符提交文件。")
            submission_df = pd.DataFrame({'row_id': test_df_check['row_id'], 'rule_violation': 0.5})
            submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
        else:
            run_parallel_inference_and_create_submission()
    except FileNotFoundError:
        print(f"错误: 测试文件 '{TEST_FILE_PATH}' 未找到。请确保该文件存在。")
        # 创建一个空的提交文件以避免 Kaggle 报错
        pd.DataFrame({'row_id': [], 'rule_violation': []}).to_csv(SUBMISSION_FILE_PATH, index=False)

Overwriting predicter_script.py


In [18]:
!python predicter_script.py

测试文件为空，生成占位符提交文件。


# 全局Rank会比Rule内Rank更好吗？

In [19]:
import pandas as pd
import numpy as np

q42 = pd.read_csv('/kaggle/working/submission_42.csv')
q43 = pd.read_csv('/kaggle/working/submission_43.csv')
q44 = pd.read_csv('/kaggle/working/submission_44.csv')
q45 = pd.read_csv('/kaggle/working/submission_45.csv')
#l = pd.read_csv('submission_qwen3.csv')
#m = pd.read_csv('submission_qwen14b.csv')
# d = pd.read_csv('submission_deberta.csv')
f = pd.read_csv('submission_faiss.csv')


rq42 = q42['rule_violation'].rank(method='average') / (len(q42)+1)
rq43 = q43['rule_violation'].rank(method='average') / (len(q43)+1)
rq44 = q44['rule_violation'].rank(method='average') / (len(q44)+1)
rq45 = q45['rule_violation'].rank(method='average') / (len(q45)+1)
#rl = l['rule_violation'].rank(method='average') / (len(l)+1)
#rm = m['rule_violation'].rank(method='average') / (len(m)+1)
# rd = d['rule_violation'].rank(method='average') / (len(d)+1)
rf = f['rule_violation'].rank(method='average') / (len(f)+1)


blend = 0.85*(0.25*rq42 + 0.25*rq43 + 0.25*rq44 + 0.25*rq45) + 0.15*rf
q42['rule_violation'] = blend
q42.to_csv('/kaggle/working/submission.csv', index=False)

# Rule内Rank

In [20]:
# import pandas as pd
# import numpy as np

# print("开始执行组内排名集成流程...")

# # --- 步骤 1: 定义文件路径 ---
# # 包含 'rule' 分组信息的原始测试文件
# test_info_path = '/kaggle/input/jigsaw-agile-community-rules/test.csv'
# # test_info_path = '/kaggle/working/test.csv'
# # 您已经生成好的 submission 文件（可以灵活添加或移除）

# submission_paths = {
#     42: '/kaggle/working/submission_42.csv',
#     43: '/kaggle/working/submission_43.csv',
#     44: '/kaggle/working/submission_44.csv',
#     45: '/kaggle/working/submission_45.csv',
# }
# # 最终输出的集成文件
# final_submission_path = '/kaggle/working/submission.csv'

# # --- 步骤 2: 加载数据并合并 ---
# print("加载 test.csv 以获取 'rule' 分组信息...")
# # 只需要 row_id 和 rule 列来进行分组
# test_info_df = pd.read_csv(test_info_path, usecols=['row_id', 'rule'])

# # 获取所有 seed 并排序，以确保一致性
# sorted_seeds = sorted(submission_paths.keys())
# if not sorted_seeds:
#     raise ValueError("没有可用的 submission 文件！")

# # 选择第一个 seed 作为基础（灵活，如果移除某个 seed，不会硬编码出错）
# base_seed = sorted_seeds[0]
# print(f"加载基础文件: {submission_paths[base_seed]} (使用第一个可用 seed: {base_seed})")
# ensemble_df = pd.read_csv(submission_paths[base_seed])
# # 重命名预测列，以区分不同模型的预测
# ensemble_df.rename(columns={'rule_violation': f'pred_{base_seed}'}, inplace=True)

# # 循环加载并合并其余的 submission 文件
# for seed in sorted_seeds[1:]:
#     print(f"加载并合并文件: {submission_paths[seed]}")
#     sub_df = pd.read_csv(submission_paths[seed])
#     # 使用 pd.merge 合并，确保 row_id 对齐
#     ensemble_df = pd.merge(
#         ensemble_df,
#         sub_df,
#         on='row_id',
#         suffixes=('', f'_{seed}')  # 添加后缀以避免列名冲突
#     )
#     # 为新合并的列重命名
#     ensemble_df.rename(columns={'rule_violation': f'pred_{seed}'}, inplace=True)

# # 将 'rule' 信息合并到集成 DataFrame 中
# ensemble_df = pd.merge(ensemble_df, test_info_df, on='row_id')
# print("\n所有数据已加载并合并。合并后数据预览:")
# print(ensemble_df.head())

# # --- 步骤 3: 组内排名转换 ---
# pred_cols = [f'pred_{seed}' for seed in sorted_seeds]
# rank_cols = []
# print("\n开始进行组内排名转换...")
# for col in pred_cols:
#     rank_col_name = col.replace('pred_', 'rank_')
#     print(f" 正在处理列: {col} -> {rank_col_name}")
#     # 使用 groupby('rule').rank(pct=True)
#     # 这会在每个 'rule' 组内独立计算排名，并将排名转换为百分位(0-1之间)
#     # pct=True 的好处是直接将排名标准化，非常适合后续求平均
#     ensemble_df[rank_col_name] = ensemble_df.groupby('rule')[col].rank(method='average', pct=True)
#     rank_cols.append(rank_col_name)
# print("\n排名转换完成。数据预览:")
# print(ensemble_df[['row_id', 'rule'] + rank_cols].head())

# # --- 步骤 4: 确定所有 rule 并集成排名生成最终结果 ---
# print("\n确定按照小写排序后的所有 rule...")
# # 获取唯一的 rule 值，按照小写排序
# unique_rules = ensemble_df['rule'].unique()
# sorted_unique_rules = sorted(unique_rules, key=str.lower)
# print(f"总共有 {len(sorted_unique_rules)} 个 rule: {sorted_unique_rules}")

# # 定义每个 rule 的集成 seeds（灵活配置，后续实验可直接修改此字典）
# # 假设有6个 rule，按照排序顺序明确指定每个的 seeds
# # rule0: sorted_unique_rules[0], rule1: sorted_unique_rules[1], 以此类推
# rule_to_seeds = {
#     sorted_unique_rules[0]: [42,43,44,45],  # rule0 只集成 42,43,44
#     sorted_unique_rules[1]: [42,43,44,45],  # rule1 集成所有
#     sorted_unique_rules[2]: [42,43,44,45],  # rule2 集成所有
#     sorted_unique_rules[3]: [42,43,44,45],  # rule3 集成所有
#     sorted_unique_rules[4]: [42,43,44,45],  # rule4 集成所有
#     sorted_unique_rules[5]: [42,43,44,45],  # rule5 集成所有
# }

# # 打印每个 rule 的集成 seeds
# print("\n每个 rule 集成的 seed:")
# for rule, seeds in rule_to_seeds.items():
#     print(f"{rule}: {', '.join(map(str, seeds))}")

# # 生成每个 rule 对应的 rank 列
# rule_to_rank_cols = {
#     rule: [f'rank_{seed}' for seed in seeds if f'rank_{seed}' in rank_cols]
#     for rule, seeds in rule_to_seeds.items()
# }

# # 检查是否所有 seeds 都可用
# for rule, cols in rule_to_rank_cols.items():
#     if not cols:
#         raise ValueError(f"Rule '{rule}' 的 seeds 未加载任何 rank 列！")

# print("\n根据 rule 进行平均集成...")
# # 使用 apply 计算每个行的平均值（灵活，支持不同 rule 用不同列）
# ensemble_df['rule_violation'] = ensemble_df.apply(
#     lambda row: ensemble_df.loc[row.name, rule_to_rank_cols[row['rule']]].mean(),
#     axis=1
# )

# # --- 步骤 5: 创建并保存最终的 submission 文件 ---
# print("生成最终的 submission.csv 文件...")
# submission_df = ensemble_df[['row_id', 'rule_violation']]
# submission_df.to_csv(final_submission_path, index=False)
# print("\n========================================================")
# print(f"集成提交文件 '{final_submission_path}' 已成功生成！")
# print("文件预览:")
# print(submission_df.head())
# print("========================================================")

----------------